# mt103-extractor-lora: LoRA training + ID/OOD evals on a Kaggle GPU

Trains a LoRA adapter for **Qwen2.5-0.5B-Instruct** (the model the repo serves through Ollama) on the
format-diverse SYNTHETIC training set **v2** (`data/generate_v2.py`, 4,800 train examples), then scores it with
the repo's own scorer on:

* the v1 test split (300, in-distribution for the original templates),
* the v2 test split (600),
* the held-out OOD set (100, `data/generate_ood.py`; no v2 template is derived from it),

each with and without the deterministic rules in `postprocess.py`. Outputs (in `/kaggle/working`):
`adapter.zip`, `metrics.json`, `results.zip` (eval JSONs, raw outputs, train log, loss curve).

Generated from the repo by `notebooks/build_notebook.py`; do not edit by hand. Settings are in the next cell.

In [ ]:
# ---- settings (defaults = what the repo's docs/kaggle.md describes) ----
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # the served model; "Qwen/Qwen2.5-1.5B-Instruct" also works (slower)
EPOCHS = 2                                   # over 4,800 v2 train examples
EXTRA_TRAIN_ARGS = "--no-qlora --batch-size 4 --grad-accum 4"   # fp16 LoRA, effective batch 16 (fits T4/P100)
EVAL_BATCH = 16

In [ ]:
!pip install -q "transformers==5.24.0" "peft==0.22.0" "trl==1.18.2" "datasets" "accelerate" "pydantic>=2" pyyaml matplotlib
import os, subprocess
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # one GPU is enough; avoids naive model-parallel across 2 GPUs
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
import torch
assert torch.cuda.is_available(), "No GPU: in the notebook settings choose Accelerator = GPU (T4 or P100)"

In [ ]:
%cd /kaggle/working

In [ ]:
import os
for d in ['mt103/data/v2', 'mt103/evals', 'mt103/train', 'mt103/models', 'mt103/results']:
    os.makedirs(d, exist_ok=True)

## Project source (written from the repo)

In [ ]:
%%writefile mt103/schema.py
"""Pydantic schema for the subset of SWIFT MT103 fields used by this project.

Fields (tag -> key):
  20  sender_reference        23B bank_operation_code
  32A value_date / currency / amount
  50K ordering_customer       52A ordering_institution (BIC)
  57A account_with_institution (BIC)
  59  beneficiary             70  remittance_info
  71A charges (OUR/SHA/BEN)

An extraction result is {"fields": MT103Fields-like dict, "errors": [str, ...]}.
Fields that are missing or impossible in the source text are null, and impossible
values are reported in `errors` (never invented).
"""
from __future__ import annotations

import datetime as dt
import re
from typing import Literal, Optional

from pydantic import BaseModel, ConfigDict, Field, field_validator

# ISO 4217 active alphabetic codes (complete list as of 2024, minus funds/precious metals).
ISO4217 = frozenset("""
AED AFN ALL AMD ANG AOA ARS AUD AWG AZN BAM BBD BDT BGN BHD BIF BMD BND BOB BRL BSD BTN BWP BYN
BZD CAD CDF CHF CLP CNY COP CRC CUP CVE CZK DJF DKK DOP DZD EGP ERN ETB EUR FJD FKP GBP GEL GHS
GIP GMD GNF GTQ GYD HKD HNL HTG HUF IDR ILS INR IQD IRR ISK JMD JOD JPY KES KGS KHR KMF KPW KRW
KWD KYD KZT LAK LBP LKR LRD LSL LYD MAD MDL MGA MKD MMK MNT MOP MRU MUR MVR MWK MXN MYR MZN NAD
NGN NIO NOK NPR NZD OMR PAB PEN PGK PHP PKR PLN PYG QAR RON RSD RUB RWF SAR SBD SCR SDG SEK SGD
SHP SLE SOS SRD SSP STN SVC SYP SZL THB TJS TMT TND TOP TRY TTD TWD TZS UAH UGX USD UYU UZS VES
VND VUV WST XAF XCD XOF XPF YER ZAR ZMW ZWL
""".split())

BIC_RE = re.compile(r"^[A-Z]{4}[A-Z]{2}[A-Z0-9]{2}([A-Z0-9]{3})?$")
REF_RE = re.compile(r"^[A-Za-z0-9/\-?:().,'+ ]{1,16}$")
BANK_OP_CODES = ("CRED", "CRTS", "SPAY", "SPRI", "SSTD")
FIELD_KEYS = (
    "sender_reference", "bank_operation_code", "value_date", "currency", "amount",
    "ordering_customer", "ordering_institution", "account_with_institution",
    "beneficiary", "remittance_info", "charges",
)
REQUIRED_KEYS = ("sender_reference", "value_date", "currency", "amount",
                 "ordering_customer", "beneficiary", "charges")
# Machine-readable error codes. Each `errors` entry is "<CODE>: <field>: <detail>", e.g.
# "INVALID_CURRENCY: currency: USX is not ISO-4217". Scoring uses (CODE, field) only.
ERROR_CODES = ("INVALID_CURRENCY", "INVALID_DATE", "INVALID_BIC", "NON_POSITIVE_AMOUNT",
               "INVALID_CHARGES", "MISSING_REQUIRED")


class Party(BaseModel):
    model_config = ConfigDict(extra="forbid")
    account: Optional[str] = None
    name: str = Field(min_length=1, max_length=140)
    address: Optional[str] = None


class MT103Fields(BaseModel):
    """All fields optional: an extractor may legitimately leave a field null."""
    model_config = ConfigDict(extra="forbid")

    sender_reference: Optional[str] = Field(None, description=":20:")
    bank_operation_code: Optional[Literal["CRED", "CRTS", "SPAY", "SPRI", "SSTD"]] = Field(None, description=":23B:")
    value_date: Optional[str] = Field(None, description=":32A: date, ISO YYYY-MM-DD")
    currency: Optional[str] = Field(None, description=":32A: ISO-4217")
    amount: Optional[float] = Field(None, description=":32A: amount, > 0")
    ordering_customer: Optional[Party] = Field(None, description=":50K:")
    ordering_institution: Optional[str] = Field(None, description=":52A: BIC")
    account_with_institution: Optional[str] = Field(None, description=":57A: BIC")
    beneficiary: Optional[Party] = Field(None, description=":59:")
    remittance_info: Optional[str] = Field(None, max_length=140, description=":70:")
    charges: Optional[Literal["OUR", "SHA", "BEN"]] = Field(None, description=":71A:")

    @field_validator("sender_reference")
    @classmethod
    def _ref(cls, v):
        if v is not None and not REF_RE.match(v):
            raise ValueError("sender_reference must be 1-16 SWIFT x-chars")
        return v

    @field_validator("value_date")
    @classmethod
    def _date(cls, v):
        if v is not None:
            dt.date.fromisoformat(v)  # raises on bad format / impossible date
            if not re.match(r"^\d{4}-\d{2}-\d{2}$", v):
                raise ValueError("value_date must be YYYY-MM-DD")
        return v

    @field_validator("currency")
    @classmethod
    def _ccy(cls, v):
        if v is not None and v not in ISO4217:
            raise ValueError(f"{v} is not an ISO-4217 currency")
        return v

    @field_validator("amount")
    @classmethod
    def _amt(cls, v):
        if v is not None and not v > 0:
            raise ValueError("amount must be positive")
        return v

    @field_validator("ordering_institution", "account_with_institution")
    @classmethod
    def _bic(cls, v):
        if v is not None and not BIC_RE.match(v):
            raise ValueError(f"{v} is not a valid BIC")
        return v


class Extraction(BaseModel):
    model_config = ConfigDict(extra="forbid")
    fields: MT103Fields
    errors: list[str] = Field(default_factory=list)

    @field_validator("errors")
    @classmethod
    def _codes(cls, v):
        for e in v:
            if e.split(":", 1)[0].strip() not in ERROR_CODES:
                raise ValueError(f"unknown error code in {e!r}")
        return v


def error_keys(errors) -> set[str]:
    """Normalise an errors list to a set of "CODE:field" keys (detail text ignored)."""
    out = set()
    if not isinstance(errors, list):
        return out
    for e in errors:
        if not isinstance(e, str):
            continue
        parts = [x.strip() for x in e.split(":")]
        code = parts[0].upper().replace(" ", "_")
        field = parts[1].split()[0].lower() if len(parts) > 1 and parts[1] else ""
        out.add(f"{code}:{field}")
    return out

In [ ]:
%%writefile mt103/prompts.py
"""Prompt construction and output parsing shared by training, eval and serving."""
from __future__ import annotations

import json
import re

SYSTEM = """You convert messy payment instructions (emails, chats, partial SWIFT text, tickets) into SWIFT MT103 JSON.
Return ONLY one JSON object, no prose, with exactly this shape:
{"fields":{"sender_reference":str|null,"bank_operation_code":"CRED"|"CRTS"|"SPAY"|"SPRI"|"SSTD"|null,"value_date":"YYYY-MM-DD"|null,"currency":ISO-4217 str|null,"amount":number|null,"ordering_customer":{"account":str|null,"name":str,"address":str|null}|null,"ordering_institution":BIC|null,"account_with_institution":BIC|null,"beneficiary":{"account":str|null,"name":str,"address":str|null}|null,"remittance_info":str|null,"charges":"OUR"|"SHA"|"BEN"|null},"errors":[str]}
Rules:
- 20=sender_reference, 23B=bank_operation_code, 32A=value_date+currency+amount, 50K=ordering_customer, 52A=ordering_institution (sender's bank BIC), 57A=account_with_institution (beneficiary's bank BIC), 59=beneficiary, 70=remittance_info, 71A=charges.
- Numeric dates are day-first (dd/mm/yyyy). SWIFT 6-digit dates are YYMMDD. Output dates as YYYY-MM-DD.
- Amounts: plain number. Indian grouping 1,25,000.50 = 125000.5; "2.5 lakh" = 250000; European 1.234,56 = 1234.56. Symbols: $=USD, €=EUR, £=GBP, Rs/₹=INR, ¥=JPY.
- Charges: ours/we pay=OUR, shared=SHA, beneficiary pays=BEN.
- Copy names, accounts, addresses and references exactly as written. A field not in the text is null. Never invent values.
- If a value is impossible, set the field to null and add an error "CODE: field: detail". Codes: INVALID_CURRENCY (not ISO-4217), INVALID_DATE (not a real date), INVALID_BIC (not 8 or 11 chars AAAABBCC[XXX]), NON_POSITIVE_AMOUNT (zero or negative), INVALID_CHARGES (not OUR/SHA/BEN), MISSING_REQUIRED (value_date, beneficiary or charges not stated).
- errors is [] when everything is fine."""

# Fixed few-shot ids from the TRAIN split (email, chat with Indian amount format, partial SWIFT, ticket with an error).
FEWSHOT_IDS = ("syn-00004", "syn-00012", "syn-00000", "syn-00005")


def target_str(target: dict) -> str:
    """Canonical compact JSON for a target (what the fine-tuned model is trained to emit)."""
    return json.dumps(target, ensure_ascii=False, separators=(",", ":"))


def chat_messages(text: str, fewshot: list[dict] | None = None) -> list[dict]:
    msgs = [{"role": "system", "content": SYSTEM}]
    for ex in fewshot or []:
        msgs.append({"role": "user", "content": ex["text"]})
        msgs.append({"role": "assistant", "content": target_str(ex["target"])})
    msgs.append({"role": "user", "content": text})
    return msgs


_FENCE = re.compile(r"```(?:json)?\s*(.*?)```", re.S)


def parse_output(raw: str):
    """Return (obj or None). Accepts fenced code blocks and leading/trailing prose."""
    if raw is None:
        return None
    s = raw.strip()
    m = _FENCE.search(s)
    if m:
        s = m.group(1).strip()
    try:
        return json.loads(s)
    except Exception:
        pass
    start = s.find("{")
    if start < 0:
        return None
    depth, in_str, esc = 0, False, False
    for i in range(start, len(s)):
        c = s[i]
        if in_str:
            if esc:
                esc = False
            elif c == "\\":
                esc = True
            elif c == '"':
                in_str = False
            continue
        if c == '"':
            in_str = True
        elif c == "{":
            depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(s[start:i + 1])
                except Exception:
                    return None
    return None


# Short system prompt used for the FINE-TUNED model (training, Ollama Modelfile, eval with
# --prompt ft). The format and error rules are learned from data, so the prompt stays tiny:
# that is what makes the fine-tuned model much cheaper per request than few-shot prompting.
FT_SYSTEM = "Extract the payment as MT103 JSON {fields, errors}. Null for absent fields; flag impossible values in errors."


def ft_messages(text: str) -> list[dict]:
    return [{"role": "system", "content": FT_SYSTEM}, {"role": "user", "content": text}]

In [ ]:
%%writefile mt103/postprocess.py
"""Deterministic post-processing of a model's extraction, checked against the source text.

It is model-agnostic (base, fine-tuned or API output) and uses no dataset-specific lists: only the
schema's own rules (ISO-4217 list, BIC shape, real calendar dates, amount > 0, OUR/SHA/BEN) and the
source text. Every rule either normalises a value, or removes something that cannot be right.

  R1 shape      keep only schema keys; "", "null", "n/a", "none" -> null; strip strings.
  R2 normalise  currency symbol / case -> ISO code; amount strings -> number (Indian lakh/crore and
                1,25,000 grouping, European 1.234,56, Swiss 1'234.50, spaces); date strings -> ISO
                (numeric dates are day-first, as in the prompt); charges words -> OUR/SHA/BEN;
                BICs and op codes upper-case.
  R3 validate   an impossible value is set to null. An error is added only if that impossible value
                literally occurs in the source text (so the model's own garbage is not reported as a
                problem with the payment).
  R4 ground     a BIC, op code, currency, amount, date, account or reference that does not occur in
                the source text is an invention and is set to null. An amount whose digit string
                matches exactly one amount in the text is replaced by that amount, parsed properly
                (fixes thousands/decimal separator misreads such as 4565.3955 for 4,565,395.50).
                An amount that only occurs negated in the text (-500, (500)) is null + NON_POSITIVE_AMOUNT.
  R5 errors     canonical "CODE: field: detail"; unknown codes and impossible code/field pairs are
                dropped; an error on a field that holds a valid value is dropped (by the schema
                contract a flagged field is null); an error that calls a valid, grounded value
                impossible is dropped and the value restored (unless another field already holds it).
  R6 add flags  value_date null with no error: INVALID_DATE if the text contains an impossible date,
                else MISSING_REQUIRED if the text contains no date at all. charges null with no error
                and no charges wording in the text: MISSING_REQUIRED.

Contract (tested): postprocess(gold, text) == gold for every gold target, i.e. the rules never
damage a correct answer.
"""
from __future__ import annotations

import calendar
import copy
import datetime as dt
import re

from schema import BANK_OP_CODES, BIC_RE, ERROR_CODES, FIELD_KEYS, ISO4217

PARTY_KEYS = ("account", "name", "address")
NULLISH = {"", "null", "none", "n/a", "na", "nil", "-", "unknown", "not stated", "not provided"}

SYMBOLS = [("US$", "USD"), ("$", "USD"), ("€", "EUR"), ("£", "GBP"), ("₹", "INR"), ("¥", "JPY")]
CCY_WORDS = {"RS": "INR", "RS.": "INR", "RUPEES": "INR", "RUPEE": "INR", "DOLLARS": "USD", "EUROS": None}
CHARGE_MAP = {"OUR": "OUR", "OURS": "OUR", "SHA": "SHA", "SHARED": "SHA", "BEN": "BEN",
              "BENEFICIARY": "BEN", "BENEFICIARY PAYS": "BEN"}
FIXED_FIELD = {"INVALID_CURRENCY": "currency", "INVALID_DATE": "value_date",
               "NON_POSITIVE_AMOUNT": "amount", "INVALID_CHARGES": "charges"}
MONTHS = {m.lower(): i for i, m in enumerate(calendar.month_abbr) if m}
MON_RE = r"(jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)[a-z]*\.?"


# ----------------------------------------------------------------------------- helpers
def _clean(v):
    if isinstance(v, str):
        s = re.sub(r"\s+", " ", v).strip()
        return None if s.casefold() in NULLISH else s
    return v


def _alnum(s: str) -> str:
    return re.sub(r"[^0-9a-z]", "", s.casefold())


def _norm_text(s: str) -> str:
    return re.sub(r"\s+", " ", s).casefold()


def _token_in(token: str, text_upper: str) -> bool:
    if re.search(r"(?<![A-Z0-9])" + re.escape(token.upper()) + r"(?![A-Z0-9])", text_upper):
        return True
    # SWIFT 32A packs date, currency and amount together: ':32A:260905INR2681837,84'
    return any(m.group("ccy") == token.upper() for m in SWIFT_32A_FULL.finditer(text_upper))


def _ccy_grounded(code: str, text_upper: str) -> bool:
    """The ISO code, or a symbol / word that means it, occurs in the text."""
    return (_token_in(code, text_upper) or any(c == code and sym in text_upper for sym, c in SYMBOLS)
            or (code == "INR" and re.search(r"\bRS\.?(?![A-Z])", text_upper) is not None))


# ----------------------------------------------------------------------------- dates
def _mk(y: int, m: int, d: int):
    if y < 100:
        y += 2000
    try:
        return dt.date(y, m, d)
    except ValueError:
        return None


_B = r"(?<![\w./-])"  # not inside a longer number (1.234.567) or a reference (PAY-2026-01)
_E = r"(?![\w/-]|\.\d)"
DATE_PATTERNS = [
    # (regex, groups -> (y, m, d)); numeric dates are day-first, 4-digit-year-first are Y-M-D
    (re.compile(_B + r"(\d{4})[-/.](\d{1,2})[-/.](\d{1,2})" + _E), lambda g: (g[0], g[1], g[2])),
    (re.compile(_B + r"(\d{1,2})[-/.](\d{1,2})[-/.](\d{4}|\d{2})" + _E), lambda g: (g[2], g[1], g[0])),
    (re.compile(_B + r"(20\d{2})(\d{2})(\d{2})" + _E), lambda g: (g[0], g[1], g[2])),
    (re.compile(r"(?i)(?<![\w])(\d{1,2})(?:st|nd|rd|th)?[\s-]+" + MON_RE + r",?[\s-]+(\d{4}|\d{2})(?!\d)"),
     lambda g: (g[2], g[1], g[0])),
    (re.compile(r"(?i)(?<![\w])" + MON_RE + r"\s+(\d{1,2})(?:st|nd|rd|th)?,?\s+(\d{4}|\d{2})(?!\d)"),
     lambda g: (g[2], g[0], g[1])),
]
SWIFT_32A = re.compile(r"(?i):32A:\s*(\d{2})(\d{2})(\d{2})(?!\d)")
SWIFT_32A_FULL = re.compile(r"(?im):32A:[ \t]*(?P<date>[^\n]*)(?<![A-Z])(?P<ccy>[A-Z]{3,5})[ \t]*[-−(]?\d")
YYMMDD = re.compile(_B + r"(2\d)(\d{2})(\d{2})" + _E)


def _month(tok: str):
    t = tok.lower().rstrip(".")[:3]
    return MONTHS.get(t)


def text_dates(text: str):
    """(valid dates, impossible date strings) mentioned in the text."""
    valid, bad = set(), []
    for rx, order in DATE_PATTERNS:
        for m in rx.finditer(text):
            y, mo, d = order(m.groups())
            mo = int(mo) if str(mo).isdigit() else _month(mo)
            if mo is None:
                continue
            x = _mk(int(y), mo, int(d))
            if x:
                valid.add(x)
            else:
                bad.append(m.group(0).strip())
    for m in SWIFT_32A.finditer(text):
        x = _mk(int(m.group(1)), int(m.group(2)), int(m.group(3)))
        (valid.add(x) if x else bad.append(m.group(1) + m.group(2) + m.group(3)))
    for m in SWIFT_32A_FULL.finditer(text):  # date glued to the currency: ':32A:17/01/2026USD1749,31'
        if m.group("date") and not re.fullmatch(r"\d{6}", m.group("date")):
            v2, b2 = text_dates(m.group("date") + " ")
            valid |= v2
            bad += [b for b in b2 if b not in bad]
    for m in YYMMDD.finditer(text):  # SWIFT-style YYMMDD, only counted when it is a real date
        x = _mk(int(m.group(1)), int(m.group(2)), int(m.group(3)))
        if x:
            valid.add(x)
    return valid, bad


def parse_date_value(v):
    """Model's value_date -> (iso or None, impossible: bool)."""
    if not isinstance(v, str):
        return None, False
    s = v.strip()
    m = re.fullmatch(r"(\d{4})-(\d{1,2})-(\d{1,2})", s)
    if m:
        x = _mk(*map(int, m.groups()))
        return (x.isoformat(), False) if x else (None, True)
    valid, bad = text_dates(s)
    if len(valid) == 1 and not bad:
        return next(iter(valid)).isoformat(), False
    return None, bool(bad)


# ----------------------------------------------------------------------------- amounts
_MULT = {"lakh": 1e5, "lakhs": 1e5, "lac": 1e5, "lacs": 1e5, "crore": 1e7, "crores": 1e7, "cr": 1e7}


def parse_number(s: str):
    """'1,25,000.50' / '1.234,56' / "1'234.50" / '12 500,00' / '2.5 lakh' / '6.9e+06' -> float."""
    if s is None:
        return None
    s = str(s).strip().replace("\u00a0", " ").replace("’", "'")
    neg = s.startswith(("-", "−")) or (s.startswith("(") and s.endswith(")"))
    s = s.strip("()-− ")
    mult = 1.0
    m = re.search(r"(?i)\s*(lakhs?|lacs?|crores?|cr)\.?$", s)
    if m:
        mult = _MULT[m.group(1).lower()]
        s = s[: m.start()]
    s = re.sub(r"[^\d.,' eE+-]", "", s).strip()
    if re.fullmatch(r"\d+(\.\d+)?[eE][+-]?\d+", s):
        x = float(s)
    else:
        s = s.replace("'", "").replace(" ", "")
        if not re.fullmatch(r"[\d.,]+", s or "x"):
            return None
        if "," in s and "." in s:
            dec = "," if s.rfind(",") > s.rfind(".") else "."
            th = "." if dec == "," else ","
            s = s.replace(th, "").replace(dec, ".")
        elif "," in s:
            parts = s.split(",")
            if len(parts) == 2 and len(parts[1]) in (1, 2):
                s = s.replace(",", ".")  # European decimal comma
            else:
                s = s.replace(",", "")
        elif s.count(".") > 1:
            s = s.replace(".", "")
        try:
            x = float(s)
        except ValueError:
            return None
    x *= mult
    return -x if neg else x


AMOUNT_RE = re.compile(
    r"(?P<neg>\(|-|−)?(?P<num>\d+(?:\.\d+)?[eE][+-]?\d+|\d{1,3}(?:[ ,.'’]\d{2,3})+(?:[.,]\d{1,2})?|\d+(?:[.,]\d+)?)"
    r"(?P<mult>\s*(?:lakhs?|lacs?|crores?|cr)\b)?(?P<close>\))?", re.I)


def _tolerance(num: str) -> float:
    """Half a unit of the last digit written: 256933 -> 0.5, 6.94919e+06 -> 5.0, 1749.31 -> 0.005."""
    m = re.fullmatch(r"(\d+)(?:\.(\d+))?[eE]([+-]?\d+)", num)
    if m:
        return max(0.005, 0.5 * 10 ** (int(m.group(3)) - len(m.group(2) or "")))
    m = re.search(r"[.,](\d{1,2})$", num)
    return max(0.005, 0.5 * 10 ** -len(m.group(1))) if m else 0.5


def text_amounts(text: str):
    """[(value, digits, negative, tolerance)] for every number-like token in the text."""
    out = []
    for m in AMOUNT_RE.finditer(text):
        prev = text[m.start() - 1] if m.start() > 0 else " "
        if prev.isalnum() and not m.group("neg") and not _after_ccy(text, m.start()):
            continue  # part of a word / reference such as TXN123
        x = parse_number(m.group("num") + (m.group("mult") or ""))
        if x is None:
            continue
        neg = bool(m.group("neg") == "(" and m.group("close")) or m.group("neg") in ("-", "−")
        if m.group("neg") in ("-", "−") and prev.isalnum():
            neg = False  # a hyphen inside a token such as 844420567-96
        tol = 0.005 if m.group("mult") else _tolerance(m.group("num"))
        out.append((x, re.sub(r"\D", "", m.group("num")).rstrip("0"), neg, tol))
    return out


def _after_ccy(text: str, i: int) -> bool:
    """True when the number is glued to a currency: 'Rs26,589.91', 'INR2681837,84' (SWIFT 32A)."""
    m = re.search(r"([A-Za-z]+)\.?$", text[max(0, i - 8):i])
    # 'rs', an ISO code, or any 3-letter upper-case code (an invalid currency such as 'GBR' is still a currency slot)
    return bool(m) and (m.group(1).lower() == "rs" or m.group(1)[-3:].upper() in ISO4217
                        or re.fullmatch(r"[A-Z]{3}", m.group(1)[-3:]) is not None)


def _digits_of(x: float) -> str:
    s = f"{x:.6f}".rstrip("0").rstrip(".")
    return re.sub(r"\D", "", s).rstrip("0")


# ----------------------------------------------------------------------------- errors
def parse_error(e):
    if not isinstance(e, str) or ":" not in e:
        return None
    parts = [p.strip() for p in e.split(":", 2)]
    code = parts[0].upper().replace(" ", "_")
    if code not in ERROR_CODES:
        return None
    raw_field = parts[1].lower() if len(parts) > 1 else ""
    detail = parts[2] if len(parts) > 2 else ""
    if code in FIXED_FIELD:
        field = FIXED_FIELD[code]
    elif code == "INVALID_BIC":
        if "order" in raw_field or "52" in raw_field or "sender" in raw_field:
            field = "ordering_institution"
        elif "account_with" in raw_field or "57" in raw_field or "benef" in raw_field or "receiv" in raw_field:
            field = "account_with_institution"
        else:
            return None
    else:  # MISSING_REQUIRED
        if "date" in raw_field:
            field = "value_date"
        elif "benef" in raw_field or raw_field.startswith("59"):
            field = "beneficiary"
        elif "charge" in raw_field or "71" in raw_field:
            field = "charges"
        else:
            return None
    return code, field, detail


def _fmt_error(code, field, detail):
    return f"{code}: {field}: {detail}" if detail else f"{code}: {field}: not stated in the text"


# ----------------------------------------------------------------------------- main
def postprocess(obj, text: str):
    """Return a new extraction dict (or obj unchanged if it is not a dict). See module docstring."""
    if not isinstance(obj, dict):
        return obj
    text = text or ""
    tu, tn, ta = text.upper(), _norm_text(text), _alnum(text)
    fin = obj.get("fields") if isinstance(obj.get("fields"), dict) else {}
    f = {k: _clean(copy.deepcopy(fin.get(k))) for k in FIELD_KEYS}
    added = []  # (code, field, detail) produced by R3/R4

    # parties (R1, R4 accounts)
    for pk in ("ordering_customer", "beneficiary"):
        p = f[pk]
        if isinstance(p, str):
            p = {"account": None, "name": p, "address": None}
        if isinstance(p, dict):
            p = {k: _clean(p.get(k)) for k in PARTY_KEYS}
            for k in PARTY_KEYS:
                if p[k] is not None and not isinstance(p[k], str):
                    p[k] = str(p[k])
            if p["account"] and _alnum(p["account"]) not in ta:
                p["account"] = None
            f[pk] = p if any(p.values()) else None
        elif p is not None:
            f[pk] = None

    # sender_reference / remittance_info
    for k in ("sender_reference", "remittance_info"):
        if f[k] is not None and not isinstance(f[k], str):
            f[k] = str(f[k])
    if f["sender_reference"] and _norm_text(f["sender_reference"]) not in tn:
        f["sender_reference"] = None

    # bank_operation_code
    if f["bank_operation_code"] is not None:
        c = str(f["bank_operation_code"]).upper()
        f["bank_operation_code"] = c if c in BANK_OP_CODES and _token_in(c, tu) else None

    # currency
    c = f["currency"]
    if c is not None:
        c = str(c).strip()
        cu = c.upper()
        c = dict(SYMBOLS).get(c, CCY_WORDS.get(cu, cu))
        if c is None or c not in ISO4217:
            if c and _token_in(c, tu):
                added.append(("INVALID_CURRENCY", "currency", f"{c} is not an ISO-4217 code"))
            c = None
        elif not _ccy_grounded(c, tu):
            c = None
        f["currency"] = c

    # amount
    a = f["amount"]
    if a is not None:
        a = parse_number(a) if isinstance(a, str) else (float(a) if isinstance(a, (int, float)) and not isinstance(a, bool) else None)
    if a is not None:
        # whole text, plus each CSV-like cell (a comma after a date must not glue it to the next amount)
        cands = text_amounts(text) + [c for cell in re.split(r'[,;\t\n"|]', text) for c in text_amounts(cell)]
        pos = [x for x, _, neg, tol in cands if not neg and abs(x - a) <= tol]
        negs = [x for x, _, neg, tol in cands if neg and abs(abs(x) - abs(a)) <= tol]
        if a <= 0:
            if any(abs(x) < 0.005 for x, _, _, _ in cands) or negs:
                added.append(("NON_POSITIVE_AMOUNT", "amount", f"{a:g} is not a positive amount"))
            a = None
        elif pos:
            pass
        elif negs:
            added.append(("NON_POSITIVE_AMOUNT", "amount", f"-{a:g} is not a positive amount"))
            a = None
        else:
            d = _digits_of(a)
            same = {x for x, digits, neg, _ in cands if digits == d and not neg and x > 0}
            a = same.pop() if len(same) == 1 else None
    f["amount"] = a

    # value_date
    dates, bad_dates = text_dates(text)
    v = f["value_date"]
    if v is not None:
        iso, impossible = parse_date_value(str(v))
        if iso is None:
            if impossible and any(_alnum(b) == _alnum(str(v)) for b in bad_dates):
                added.append(("INVALID_DATE", "value_date", f"{v} is not a real calendar date"))
            elif impossible and bad_dates:
                added.append(("INVALID_DATE", "value_date", f"{bad_dates[0]} is not a real calendar date"))
        elif dt.date.fromisoformat(iso) not in dates:
            iso = None
        f["value_date"] = iso

    # BICs
    for k in ("ordering_institution", "account_with_institution"):
        b = f[k]
        if b is None:
            continue
        b = str(b).strip().upper()
        if not BIC_RE.match(b):
            if _token_in(b, tu):
                added.append(("INVALID_BIC", k, f"{b} is not a valid BIC"))
            b = None
        elif not (_token_in(b, tu) or (len(b) == 8 and _token_in(b + "XXX", tu))):
            b = None
        f[k] = b

    # charges
    ch = f["charges"]
    if ch is not None:
        cu = re.sub(r"\s+", " ", str(ch)).strip().upper()
        mapped = CHARGE_MAP.get(re.sub(r"\s*\(.*\)$", "", cu))
        if mapped is None and not re.search(r"[+/&]", cu):
            hits = {CHARGE_MAP[t] for t in re.findall(r"[A-Z]+", cu) if t in ("OUR", "OURS", "SHA", "BEN")}
            mapped = hits.pop() if len(hits) == 1 else None
        if mapped is None:
            if _token_in(cu, tu):
                added.append(("INVALID_CHARGES", "charges", f"{ch} is not OUR/SHA/BEN"))
        f["charges"] = mapped

    # R5: model errors
    def valid_value(field, val):
        if val is None:
            return False
        if field == "currency":
            return val in ISO4217
        if field == "value_date":
            return parse_date_value(val)[0] is not None and not parse_date_value(val)[1]
        if field == "amount":
            return isinstance(val, (int, float)) and val > 0
        if field in ("ordering_institution", "account_with_institution"):
            return bool(BIC_RE.match(val))
        if field == "charges":
            return val in ("OUR", "SHA", "BEN")
        return True

    errs, seen = [], set()
    raw_errors = obj.get("errors") if isinstance(obj.get("errors"), list) else []
    for e in [parse_error(x) for x in raw_errors] + added:
        if e is None:
            continue
        code, field, detail = e
        if (code, field) in seen:
            continue
        if f.get(field) is not None:  # a flagged field must be null; the value wins if it survived R2-R4
            continue
        if code != "MISSING_REQUIRED":
            quoted = detail.split(" ")[0].strip("'\"") if detail else ""
            restored = _restore(field, quoted, f, tu, dates)
            if restored is not None:
                f[field] = restored
                continue
            if quoted and valid_value(field, quoted):
                continue  # the model called a valid value impossible; it is not in the text as such
        seen.add((code, field))
        errs.append(_fmt_error(code, field, detail))

    # R6: required fields with nothing at all in the text
    flagged = {fld for _, fld in seen}
    if f["value_date"] is None and "value_date" not in flagged:
        if bad_dates:
            errs.append(_fmt_error("INVALID_DATE", "value_date", f"{bad_dates[0]} is not a real calendar date"))
        elif not dates:
            errs.append(_fmt_error("MISSING_REQUIRED", "value_date", ""))
    if f["charges"] is None and "charges" not in flagged and not (
            re.search(r"(?i)charg|chrg|\bfees?\b|\bcosts?\b|bearer|shared", text)
            or re.search(r"\b(OUR|SHA|BEN)\b", text)):
        errs.append(_fmt_error("MISSING_REQUIRED", "charges", ""))
    return {"fields": f, "errors": errs}


def _restore(field, quoted, f, tu, dates):
    """An error that quotes a valid value which is in the text: restore the value instead."""
    if not quoted:
        return None
    q = quoted.strip(".,;")
    if field == "currency" and q.upper() in ISO4217 and _ccy_grounded(q.upper(), tu):
        return q.upper()
    if field in ("ordering_institution", "account_with_institution"):
        other = "account_with_institution" if field == "ordering_institution" else "ordering_institution"
        if BIC_RE.match(q.upper()) and _token_in(q, tu) and f.get(other) != q.upper():
            return q.upper()
    if field == "charges" and q.upper() in ("OUR", "SHA", "BEN") and _token_in(q, tu):
        return q.upper()
    return None

In [ ]:
open('mt103/data/__init__.py', 'w').close()

In [ ]:
%%writefile mt103/data/generate.py
"""Seeded generator of SYNTHETIC "messy payment text -> MT103 JSON" examples.

Everything produced here is synthetic. Names, accounts, references and amounts are
random; BICs are reused from the swift-mt103-monitor simulator's bank list plus a few
other well-known public BICs. No real payment data is used.

Usage:
    python data/generate.py --n 3000 --seed 103 --out data/
Writes data/{train,val,test}.jsonl (80/10/10) and data/MANIFEST.json.
"""
from __future__ import annotations

import argparse
import datetime as dt
import hashlib
import json
import random
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from schema import ISO4217, Extraction  # noqa: E402

# Bank list from swift-mt103-monitor/swift/producer.go, extended.
BANKS_SRC = ["DEUTDEDB", "CHASUS33", "BARCGB22", "BNPAFRPP", "HSBCHKHH"]
BANKS_EXTRA = ["SBININBB", "HDFCINBB", "ICICINBBXXX", "UTIBINBB", "CITIUS33", "DEUTDEFF",
               "BOFAUS3N", "SCBLSGSG", "MHCBJPJT", "UBSWCHZH80A", "INGBNL2A", "COBADEFFXXX"]
BANKS = BANKS_SRC + BANKS_EXTRA
BAD_BICS = ["HDFC123", "CHASUS", "BARC-GB22", "1234INBB", "DEUTDEDBX", "SBIN IN BB", "HSBCHK"]

# currency -> (weight, typical amount range)
CCY = {"USD": (40, (10, 10000)), "EUR": (14, (50, 250000)), "GBP": (8, (50, 150000)),
       "INR": (20, (1000, 9000000)), "SGD": (4, (100, 200000)), "JPY": (4, (10000, 9000000)),
       "CHF": (3, (100, 100000)), "AED": (4, (500, 400000)), "HKD": (3, (500, 500000))}
CCY_SYMBOL = {"USD": ["$", "US$"], "EUR": ["€"], "GBP": ["£"], "INR": ["Rs.", "₹", "INR", "Rs"],
              "JPY": ["¥"]}
BAD_CCY = ["USX", "EUD", "RSS", "GBR", "IND", "DOL", "UDS", "ERU"]

FIRST = ["Aarav", "Priya", "Rohan", "Ananya", "Vikram", "Meera", "John", "Sarah", "Lukas", "Chloe",
         "Kenji", "Fatima", "Omar", "Elena", "Arjun", "Divya", "Marco", "Aisha", "Wei", "Nikhil"]
LAST = ["Sharma", "Iyer", "Patel", "Reddy", "Kapoor", "Smith", "Müller", "Dubois", "Tanaka",
        "Khan", "Rossi", "Nair", "Chen", "Gupta", "Fernandes", "Menon", "Brown", "Schmidt"]
CO_A = ["Arcadia", "Bluepeak", "Coromandel", "Deccan", "Everest", "Falcon", "Ganges", "Helix",
        "Indus", "Juniper", "Kestrel", "Lotus", "Meridian", "Nilgiri", "Orion", "Saffron",
        "Tata-Vale", "Zenith", "Northwind", "Harbor"]
CO_B = ["Textiles", "Logistics", "Pharma", "Steel", "Software", "Exports", "Agro", "Capital",
        "Trading", "Components", "Foods", "Shipping", "Analytics", "Motors"]
CO_C = ["Pvt Ltd", "Ltd", "GmbH", "LLC", "Inc", "SA", "Pte Ltd", "LLP", "BV", "PLC"]
CITY = ["Mumbai 400001 IN", "Bengaluru 560001 IN", "Pune 411001 IN", "Chennai 600002 IN",
        "Frankfurt 60311 DE", "London EC2V 7HN GB", "New York NY 10005 US", "Paris 75009 FR",
        "Singapore 048583 SG", "Hong Kong HK", "Dubai AE", "Zurich 8001 CH", "Tokyo 100-0005 JP"]
STREET = ["12 MG Road", "4th Floor, Nariman Point", "221 Park Street", "Plot 7, MIDC",
          "Taunusanlage 12", "1 Churchill Place", "270 Park Avenue", "16 Bd des Italiens",
          "8 Marina Blvd", "Unit 5B, Hi-Tech Park"]
REMIT = ["INV {n} consulting fees", "Invoice {n} / spare parts", "PO {n} advance payment",
         "Salary {mon} 2026", "Rent {mon} 2026 office", "Refund order {n}",
         "Freight charges BL {n}", "Licence fee Q{q} 2026", "Settlement of INV {n} and INV {n2}",
         "Tuition fees semester {q}", "Dividend payout {mon}", "Software subscription {n}"]
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
MONTHS_LONG = ["January", "February", "March", "April", "May", "June", "July", "August",
               "September", "October", "November", "December"]
CHARGE_WORDS = {"OUR": ["OUR", "our", "ours", "OUR (we pay all charges)", "all charges on us"],
                "SHA": ["SHA", "sha", "shared", "SHA - shared", "split charges (SHA)"],
                "BEN": ["BEN", "ben", "beneficiary pays", "BEN (charges to beneficiary)"]}
BAD_CHARGES = ["FULL", "NONE", "OURS+BEN", "XYZ", "HALF"]
TYPOS = {"beneficiary": ["benificiary", "beneficary", "bene", "benef."], "amount": ["amt", "amout", "amnt"],
         "please": ["pls", "plz", "please"], "account": ["acct", "a/c", "acc"],
         "reference": ["ref", "refrence", "ref no"], "payment": ["pymt", "paymnt", "payment"],
         "charges": ["charges", "chrgs", "charge"], "value date": ["value date", "val dt", "VD", "value dt"]}


def indian_format(x: float, decimals: bool = True) -> str:
    s = f"{x:.2f}"
    whole, frac = s.split(".")
    if len(whole) > 3:
        head, tail = whole[:-3], whole[-3:]
        groups = []
        while len(head) > 2:
            groups.insert(0, head[-2:])
            head = head[:-2]
        if head:
            groups.insert(0, head)
        whole = ",".join(groups) + "," + tail
    return f"{whole}.{frac}" if decimals or frac != "00" else whole


class Gen:
    def __init__(self, seed: int):
        self.r = random.Random(seed)

    # ---------- primitives ----------
    def w(self, key: str) -> str:
        return self.r.choice(TYPOS[key]) if self.r.random() < 0.5 else key

    def company(self) -> str:
        r = self.r
        return f"{r.choice(CO_A)} {r.choice(CO_B)} {r.choice(CO_C)}"

    def person(self) -> str:
        return f"{self.r.choice(FIRST)} {self.r.choice(LAST)}"

    def account(self) -> str:
        r = self.r
        k = r.random()
        if k < 0.4:
            return "".join(r.choice("0123456789") for _ in range(r.choice([11, 12, 14, 15])))
        if k < 0.75:
            cc = r.choice(["DE", "GB", "FR", "NL", "CH"])
            return cc + "".join(r.choice("0123456789") for _ in range(r.choice([18, 20])))
        return "".join(r.choice("0123456789") for _ in range(9)) + "-" + str(r.randint(10, 99))

    def party(self, need_account: bool = True) -> dict:
        r = self.r
        name = self.company() if r.random() < 0.7 else self.person()
        acct = self.account() if (need_account or r.random() < 0.7) else None
        addr = f"{r.choice(STREET)}, {r.choice(CITY)}" if r.random() < 0.55 else None
        return {"account": acct, "name": name, "address": addr}

    def ref(self) -> str:
        r = self.r
        k = r.random()
        if k < 0.4:
            return f"TXN{r.randint(0, 999999):06d}"  # same pattern as swift-mt103-monitor
        if k < 0.7:
            return f"FT26{r.randint(10000000, 99999999)}"[:16]
        if k < 0.9:
            return f"PAY-{r.randint(2025, 2026)}-{r.randint(1, 99999):05d}"
        return f"OPS/{r.choice(MONTHS).upper()}/{r.randint(1, 999)}"

    def amount(self, ccy: str) -> float:
        lo, hi = CCY[ccy][1]
        x = round(self.r.uniform(lo, hi), 2)
        if ccy == "JPY":
            x = float(int(x))
        if self.r.random() < 0.25:
            x = float(round(x, -2)) or 100.0
        return x

    def fmt_amount(self, x: float, ccy: str) -> str:
        r = self.r
        k = r.random()
        if ccy == "INR" and k < 0.55:
            if x >= 100000 and r.random() < 0.2 and round(x / 100000, 2) * 100000 == x:
                return f"{x / 100000:g} lakh"
            return indian_format(x, decimals=r.random() < 0.7)
        if ccy == "EUR" and k < 0.25:
            s = f"{x:,.2f}"
            return s.replace(",", "_").replace(".", ",").replace("_", ".")
        if k < 0.6:
            return f"{x:,.2f}"
        if k < 0.85:
            return f"{x:.2f}" if r.random() < 0.6 else f"{x:g}"
        return f"{x:,.2f}".replace(",", " ")

    def fmt_ccy(self, ccy: str) -> str:
        r = self.r
        if ccy in CCY_SYMBOL and r.random() < 0.35:
            return r.choice(CCY_SYMBOL[ccy])
        return ccy if r.random() < 0.8 else ccy.lower()

    def date(self) -> dt.date:
        return dt.date(2026, 1, 1) + dt.timedelta(days=self.r.randint(0, 420))

    def fmt_date(self, d: dt.date) -> str:
        r = self.r
        k = r.randint(0, 7)
        return [d.isoformat(), d.strftime("%d/%m/%Y"), d.strftime("%d-%m-%Y"), d.strftime("%d.%m.%Y"),
                f"{d.day} {MONTHS[d.month - 1]} {d.year}", f"{MONTHS_LONG[d.month - 1]} {d.day}, {d.year}",
                f"{d.day:02d}-{MONTHS[d.month - 1]}-{d.strftime('%y')}", d.strftime("%y%m%d")][k]

    def bad_date(self) -> str:
        r = self.r
        y = r.choice([2026, 2027])
        return r.choice([f"31/04/{y}", f"30/02/{y}", f"{y}-13-05", f"32.01.{y}", f"31 Jun {y}",
                         f"29/02/{y}" if y == 2026 else f"31/09/{y}", f"{y}-00-12", f"31-11-{y}"])

    def remit(self) -> str:
        r = self.r
        return r.choice(REMIT).format(n=f"{r.randint(1, 9999):04d}", n2=f"{r.randint(1, 9999):04d}",
                                      mon=r.choice(MONTHS), q=r.randint(1, 4))

    # ---------- one example ----------
    def example(self, idx: int) -> dict:
        r = self.r
        ccy = r.choices(list(CCY), weights=[v[0] for v in CCY.values()])[0]
        amt = self.amount(ccy)
        d = self.date()
        f = {
            "sender_reference": self.ref(),
            "bank_operation_code": "CRED" if r.random() < 0.9 else r.choice(["SPAY", "SPRI", "SSTD", "CRTS"]),
            "value_date": d.isoformat(), "currency": ccy, "amount": amt,
            "ordering_customer": self.party(), "ordering_institution": r.choice(BANKS),
            "account_with_institution": r.choice(BANKS), "beneficiary": self.party(),
            "remittance_info": self.remit(), "charges": r.choice(["OUR", "SHA", "SHA", "BEN"]),
        }
        # surface strings (what appears in the text)
        surf = {"date": self.fmt_date(d), "ccy": self.fmt_ccy(ccy), "amt": self.fmt_amount(amt, ccy),
                "charges": r.choice(CHARGE_WORDS[f["charges"]])}
        # optional fields absent from text -> null in target, no error
        for k, p in (("bank_operation_code", 0.45), ("ordering_institution", 0.3),
                     ("account_with_institution", 0.3), ("remittance_info", 0.2)):
            if r.random() < p:
                f[k] = None
        errors: list[str] = []
        # impossible values (~22% of examples have at least one)
        if r.random() < 0.22:
            kind = r.choice(["ccy", "date", "bic", "amt", "charges", "missing"])
            if kind == "ccy":
                bad = r.choice(BAD_CCY); assert bad not in ISO4217
                surf["ccy"] = bad; f["currency"] = None
                errors.append(f"INVALID_CURRENCY: currency: {bad} is not an ISO-4217 code")
            elif kind == "date":
                surf["date"] = self.bad_date(); f["value_date"] = None
                errors.append(f"INVALID_DATE: value_date: {surf['date']} is not a real calendar date")
            elif kind == "bic":
                which = r.choice(["ordering_institution", "account_with_institution"])
                bad = r.choice(BAD_BICS); f[which] = None
                surf["badbic"] = (which, bad)
                errors.append(f"INVALID_BIC: {which}: {bad} is not a valid BIC")
            elif kind == "amt":
                bad = r.choice(["0.00", "-" + self.fmt_amount(amt, ccy), "0", "-" + f"{amt:.2f}"])
                surf["amt"] = bad; f["amount"] = None
                errors.append(f"NON_POSITIVE_AMOUNT: amount: {bad} is not a positive amount")
            elif kind == "charges":
                bad = r.choice(BAD_CHARGES); surf["charges"] = bad; f["charges"] = None
                errors.append(f"INVALID_CHARGES: charges: {bad} is not OUR/SHA/BEN")
            else:
                miss = r.choice(["charges", "beneficiary", "value_date"])
                f[miss] = None; surf["missing"] = miss
                errors.append(f"MISSING_REQUIRED: {miss}: not stated in the text")
        text = self.render(f, surf)
        target = {"fields": f, "errors": errors}
        Extraction.model_validate(target)  # every target must be schema-valid
        return {"id": f"syn-{idx:05d}", "synthetic": True, "text": text, "target": target}

    # ---------- renderers ----------
    def bic_s(self, f, surf, key):
        if "badbic" in surf and surf["badbic"][0] == key:
            return surf["badbic"][1]
        return f[key]

    def party_s(self, p, sep=", "):
        bits = []
        if p.get("account"):
            bits.append(p["account"] if self.r.random() < 0.5 else f"{self.w('account')} {p['account']}")
        bits.append(p["name"])
        if p.get("address"):
            bits.append(p["address"])
        return sep.join(bits)

    def render(self, f, surf) -> str:
        style = self.r.choices(["email", "chat", "swift", "ticket", "forward"], weights=[30, 25, 20, 15, 10])[0]
        return getattr(self, f"r_{style}")(f, surf).strip()

    def money(self, surf):
        c, a = surf["ccy"], surf["amt"]
        if c in ("$", "US$", "€", "£", "₹", "¥", "Rs.", "Rs"):
            return f"{c}{a}" if self.r.random() < 0.6 else f"{c} {a}"
        return f"{c} {a}" if self.r.random() < 0.7 else f"{a} {c}"

    def r_email(self, f, s):
        r, w = self.r, self.w
        oc, bn = f["ordering_customer"], f["beneficiary"]
        lines = [r.choice(["Hi team,", "Dear Ops,", "Hello,", "Good morning,", "Hi"]), ""]
        lines.append(f"{w('please').capitalize()} process the below {w('payment')} today.")
        lines.append(f"Our {w('reference')}: {f['sender_reference']}")
        lines.append(f"{w('amount').capitalize()}: {self.money(s)}")
        if s.get("missing") != "value_date":
            lines.append(f"{w('value date').capitalize()}: {s['date']}")
        lines.append(f"Debit {w('account')} / remitter: {self.party_s(oc)}")
        if f["ordering_institution"] or "badbic" in s and s["badbic"][0] == "ordering_institution":
            lines.append(f"Our bank: {self.bic_s(f, s, 'ordering_institution')}")
        if s.get("missing") != "beneficiary":
            lines.append(f"{w('beneficiary').capitalize()}: {self.party_s(bn)}")
        if f["account_with_institution"] or "badbic" in s and s["badbic"][0] == "account_with_institution":
            lines.append(f"{w('beneficiary').capitalize()} bank SWIFT: {self.bic_s(f, s, 'account_with_institution')}")
        if f["remittance_info"]:
            lines.append(f"Purpose: {f['remittance_info']}")
        if s.get("missing") != "charges":
            lines.append(f"{w('charges').capitalize()}: {s['charges']}")
        if f["bank_operation_code"]:
            lines.append(f"Op code {f['bank_operation_code']}")
        body = lines[:2] + r.sample(lines[2:], len(lines) - 2) if r.random() < 0.3 else lines
        body += ["", r.choice(["Thanks,", "Regards,", "Thx", "Best,"]), self.person()]
        return "\n".join(body)

    def r_chat(self, f, s):
        r, w = self.r, self.w
        oc, bn = f["ordering_customer"], f["beneficiary"]
        parts = [r.choice(["hey", "urgent:", "pls rush -", "@ops", "quick one"]),
                 f"send {self.money(s)} to {self.party_s(bn, ' ') if s.get('missing') != 'beneficiary' else 'the usual guys'}"]
        if f["account_with_institution"] or "badbic" in s and s["badbic"][0] == "account_with_institution":
            parts.append(f"at {self.bic_s(f, s, 'account_with_institution')}")
        parts.append(f"from {self.party_s(oc, ' ')}")
        if f["ordering_institution"] or "badbic" in s and s["badbic"][0] == "ordering_institution":
            parts.append(f"via {self.bic_s(f, s, 'ordering_institution')}")
        if s.get("missing") != "value_date":
            parts.append(f"{w('value date')} {s['date']}")
        parts.append(f"{w('reference')} {f['sender_reference']}")
        if f["remittance_info"]:
            parts.append(f"for {f['remittance_info']}")
        if s.get("missing") != "charges":
            parts.append(f"{w('charges')} {s['charges']}")
        if f["bank_operation_code"]:
            parts.append(f"({f['bank_operation_code']})")
        txt = " ".join(parts)
        return txt.lower() if r.random() < 0.3 else txt

    def r_swift(self, f, s):
        r = self.r
        oc, bn = f["ordering_customer"], f["beneficiary"]
        ccy = s["ccy"] if len(s["ccy"]) == 3 and s["ccy"].isalpha() else f["currency"] or s["ccy"]
        ccy = ccy.upper()
        amt = s["amt"]
        if not amt.startswith("-") and "lakh" not in amt:
            amt = f"{f['amount']:.2f}".replace(".", ",") if f["amount"] else amt
        date = s["date"]
        if f["value_date"] and r.random() < 0.7:
            date = dt.date.fromisoformat(f["value_date"]).strftime("%y%m%d")
        out = []
        if r.random() < 0.4 and f["ordering_institution"] and f["account_with_institution"]:
            # header only when both BICs are real, so it never leaks a BIC the target says is absent
            out.append(f"{{1:F01{f['ordering_institution'][:8]}AXXX0000000000}}{{2:I103{f['account_with_institution'][:8]}XN}}{{4:")
        out.append(f":20:{f['sender_reference']}")
        if f["bank_operation_code"]:
            out.append(f":23B:{f['bank_operation_code']}")
        if s.get("missing") != "value_date":
            out.append(f":32A:{date}{ccy}{amt}")
        else:
            out.append(f":32A:{ccy}{amt}  (date tbc)")
        out.append(":50K:" + "\n".join(x for x in [("/" + oc["account"]) if oc["account"] else None, oc["name"], oc["address"]] if x))
        if f["ordering_institution"] or "badbic" in s and s["badbic"][0] == "ordering_institution":
            out.append(f":52A:{self.bic_s(f, s, 'ordering_institution')}")
        if f["account_with_institution"] or "badbic" in s and s["badbic"][0] == "account_with_institution":
            out.append(f":57A:{self.bic_s(f, s, 'account_with_institution')}")
        if s.get("missing") != "beneficiary":
            out.append(":59:" + "\n".join(x for x in [("/" + bn["account"]) if bn["account"] else None, bn["name"], bn["address"]] if x))
        if f["remittance_info"]:
            out.append(f":70:{f['remittance_info']}")
        if s.get("missing") != "charges":
            ch = s["charges"] if s["charges"] in BAD_CHARGES else f["charges"]
            out.append(f":71A:{ch}")
        if r.random() < 0.3:  # partial paste: chopped header / trailing junk
            out.insert(0, r.choice(["pasted from SWIFT Alliance:", "msg below, pls check", "---"]))
        out.append("-}" if out[0].startswith("{1:") else r.choice(["", "-}", "...(rest cut off)"]))
        return "\n".join(out)

    def r_ticket(self, f, s):
        r = self.r
        oc, bn = f["ordering_customer"], f["beneficiary"]
        rows = [("Ticket", f"OPS-{r.randint(1000, 9999)}"), ("Ref", f["sender_reference"]),
                ("Ccy/Amt", f"{s['ccy']} | {s['amt']}")]
        if s.get("missing") != "value_date":
            rows.append(("VD", s["date"]))
        rows.append(("Ordering cust", self.party_s(oc, " ; ")))
        if f["ordering_institution"] or "badbic" in s and s["badbic"][0] == "ordering_institution":
            rows.append(("Ord inst (52A)", self.bic_s(f, s, "ordering_institution")))
        if s.get("missing") != "beneficiary":
            rows.append(("Bene", self.party_s(bn, " ; ")))
        if f["account_with_institution"] or "badbic" in s and s["badbic"][0] == "account_with_institution":
            rows.append(("AWI (57A)", self.bic_s(f, s, "account_with_institution")))
        if f["remittance_info"]:
            rows.append(("Details", f["remittance_info"]))
        if s.get("missing") != "charges":
            rows.append(("Chg", s["charges"]))
        if f["bank_operation_code"]:
            rows.append(("23B", f["bank_operation_code"]))
        sep = r.choice([": ", " = ", " - ", "\t"])
        return "\n".join(f"{k}{sep}{v}" for k, v in rows)

    def r_forward(self, f, s):
        head = "\n".join(["---------- Forwarded message ---------",
                          f"From: {self.person()} <ops{self.r.randint(1, 99)}@example.com>",
                          f"Subject: {self.r.choice(['RE: payment', 'FW: wire instructions', 'Urgent remittance', 'pymt req'])}", ""])
        inner = self.r_email(f, s) if self.r.random() < 0.5 else self.r_chat(f, s)
        return head + inner + "\n\n> earlier: pls ignore the draft I sent yesterday"


def split_ids(n: int, seed: int):
    ids = list(range(n))
    random.Random(seed + 1).shuffle(ids)
    a, b = int(n * 0.8), int(n * 0.9)
    return ids[:a], ids[a:b], ids[b:]


def generate(n: int, seed: int) -> list[dict]:
    g = Gen(seed)
    rows, seen = [], set()
    i = 0
    while len(rows) < n:
        ex = g.example(len(rows))
        i += 1
        h = hashlib.sha1(ex["text"].encode()).hexdigest()
        if h in seen:  # avoid duplicate texts across splits
            continue
        seen.add(h)
        rows.append(ex)
    return rows


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--n", type=int, default=3000)
    ap.add_argument("--seed", type=int, default=103)
    ap.add_argument("--out", default=str(Path(__file__).parent))
    a = ap.parse_args()
    rows = generate(a.n, a.seed)
    tr, va, te = split_ids(len(rows), a.seed)
    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    counts = {}
    for name, idx in (("train", tr), ("val", va), ("test", te)):
        with open(out / f"{name}.jsonl", "w", encoding="utf-8") as fh:
            for i in sorted(idx):
                fh.write(json.dumps(rows[i], ensure_ascii=False) + "\n")
        counts[name] = len(idx)
    n_err = sum(1 for r in rows if r["target"]["errors"])
    manifest = {"synthetic": True, "note": "SYNTHETIC data generated by data/generate.py; not real payments.",
                "seed": a.seed, "n": len(rows), "splits": counts, "examples_with_errors": n_err}
    (out / "MANIFEST.json").write_text(json.dumps(manifest, indent=2))
    print(json.dumps(manifest))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile mt103/data/generate_v2.py
"""Format-diverse SYNTHETIC training data (v2): many more templates, phrasings and orderings.

Everything produced here is synthetic (random names, accounts, references and amounts; public BICs).

Why: eval round 2 showed the CPU smoke model scores 94.2% on the v1 test split but 66.7% on unseen
formats (data/ood_test.jsonl): it learnt the v1 templates, e.g. it maps SWIFT fields by position and
puts a lone BIC into 57A. v2 attacks that with variety, and with cases that a template-matcher gets
wrong:
  * 10 renderer families (v1's 5 are kept for ~16% of rows): email, chat log with timestamps, SWIFT
    with extra/distractor tags (:13C: :33B: :36: :53A: :56A: :71F: :72:), aligned forms, sectioned
    forms with multi-line parties, CSV export, YAML-like block, formal letter, multilingual
    (de/fr/es/Hindi-Latin labels), long forwarded thread with quoted old instructions
  * many label synonyms per field, random field order everywhere except SWIFT, optional fields
    dropped independently, both BICs or only one of them (labelled, never positional)
  * distractors: other amounts (GST total, fee estimate, old cancelled instruction), other
    references (invoice, ticket, old ref), other dates (mail headers, call dates), IFSC codes,
    phone numbers, VAT numbers, disclaimers, signatures
  * dates: v1 formats plus "05 March 2026", "Thursday, 5 March 2026", "Mar 5, 2026", "5/3/2026",
    "05/03/26", "05.03.26", "5-Mar-2026"
  * amounts: v1 formats (without v1's lossy %g rendering) plus Indian "1,25,000/-", "12.5 lakh",
    "1.2 crore", Swiss "1'234.50", "1 234 567,89", unicode minus for negatives
  * noise: typos (transposed / dropped / doubled letters), random case, extra blank lines and spaces
  * error rate 30% (v1 22%), more MISSING_REQUIRED, and impossible values from lists that extend v1's

HELD-OUT GUARANTEE (tests/test_v2_heldout.py checks each point on the generated files):
  the OOD set (data/generate_ood.py) is NOT used or imitated. v2 has no `| key | value |` tables, no
  `key=value;` one-liners, no shuffled or lower-case SWIFT tags, no OCR letter/digit swaps, none of
  the OOD-only label names, none of the OOD date formats (ordinal days, 2026/03/05, 20260305,
  "Thu 5 Mar 26", 5.3.2026, 05-MAR-2026, "Mar. 5 2026"), none of the OOD remark sentences and none
  of the OOD impossible values. data/generate_ood.py is not modified, so data/ood_test.jsonl is
  byte-identical to the one scored in eval round 2.

Usage:
    python data/generate_v2.py --n 6000 --seed 2026 --out data/v2
Writes data/v2/{train,val,test}.jsonl (80/10/10) and data/v2/MANIFEST.json.
"""
from __future__ import annotations

import argparse
import datetime as dt
import hashlib
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from data.generate import (BAD_BICS, BAD_CCY, BAD_CHARGES, BANKS, CCY, CCY_SYMBOL, MONTHS,  # noqa: E402
                           MONTHS_LONG, Gen, indian_format, split_ids)
from schema import BIC_RE, ISO4217, Extraction  # noqa: E402

DAYS_LONG = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

BANKS_V2 = BANKS + ["KKBKINBB", "YESBINBB", "PUNBINBB", "BKIDINBB", "CNRBINBB", "NWBKGB2L", "LOYDGB2L",
                    "SOGEFRPP", "CRLYFRPP", "RABONL2U", "ABNANL2A", "DBSSSGSG", "OCBCSGSG", "BOTKJPJT",
                    "SMBCJPJT", "ANZBAU3M", "NATAAU33", "ROYCCAT2", "TDOMCATT", "NBADAEAA", "EBILAEAD",
                    "CRESCHZZ80A", "WFBIUS6S", "IRVTUS3N", "HDFCINBBXXX", "SBININBB104"]
INDIAN_BANKS = {"SBININBB", "HDFCINBB", "ICICINBBXXX", "UTIBINBB", "KKBKINBB", "YESBINBB", "PUNBINBB",
                "BKIDINBB", "CNRBINBB", "HDFCINBBXXX", "SBININBB104"}
CCY_V2 = dict(CCY, CAD=(3, (100, 200000)), AUD=(3, (100, 200000)), SAR=(2, (500, 400000)),
              ZAR=(1, (500, 900000)), SEK=(1, (500, 900000)), CNY=(2, (500, 900000)))

# impossible values: v1's lists plus new ones; all disjoint from the OOD lists (tested)
BAD_CCY_V2 = BAD_CCY + ["SGP", "JYP", "HKG", "CHN", "AUS", "UAE", "EUU", "USS"]
BAD_BICS_V2 = BAD_BICS + ["SBININ", "HDFC0001234", "CITIUS3", "BARCGB2", "DEUTDEDB1", "ABNA-NL2A", "12345678",
                          "BNP@FRPP"]
BAD_CHARGES_V2 = BAD_CHARGES + ["ALL", "NIL", "SHARE", "BENOUR", "TBD"]

FIRST_V2 = ["Aarav", "Priya", "Rohan", "Ananya", "Vikram", "Meera", "John", "Sarah", "Lukas", "Chloe",
            "Kenji", "Fatima", "Omar", "Elena", "Arjun", "Divya", "Marco", "Aisha", "Wei", "Nikhil",
            "Siddharth", "Kavya", "Rahul", "Sneha", "Ishaan", "Pooja", "Hannah", "Mateo", "Sofia",
            "Yuki", "Ahmed", "Grace", "Tomás", "Ingrid", "Ravi", "Lakshmi", "Daniel", "Amara"]
LAST_V2 = ["Sharma", "Iyer", "Patel", "Reddy", "Kapoor", "Smith", "Müller", "Dubois", "Tanaka", "Khan",
           "Rossi", "Nair", "Chen", "Gupta", "Fernandes", "Menon", "Brown", "Schmidt", "Banerjee",
           "Chatterjee", "Joshi", "Desai", "Kulkarni", "Okafor", "García", "Lindqvist", "Nakamura",
           "O'Brien", "Hassan", "Pillai", "Verma", "Agarwal", "Novak", "Silva"]
CO_A_V2 = ["Arcadia", "Bluepeak", "Coromandel", "Deccan", "Everest", "Falcon", "Ganges", "Helix", "Indus",
           "Juniper", "Kestrel", "Lotus", "Meridian", "Nilgiri", "Orion", "Saffron", "Tata-Vale", "Zenith",
           "Northwind", "Harbor", "Sahyadri", "Konkan", "Vindhya", "Aravalli", "Brightline", "Cobalt",
           "Silverleaf", "Redwood", "Alpine", "Monsoon", "Quartz", "Riverstone"]
CO_B_V2 = ["Textiles", "Logistics", "Pharma", "Steel", "Software", "Exports", "Agro", "Capital", "Trading",
           "Components", "Foods", "Shipping", "Analytics", "Motors", "Solar", "Ceramics", "Healthcare",
           "Infra", "Chemicals", "Retail", "Fintech", "Polymers"]
CO_C_V2 = ["Pvt Ltd", "Ltd", "GmbH", "LLC", "Inc", "SA", "Pte Ltd", "LLP", "BV", "PLC", "Private Limited",
           "AG", "SARL", "Corp", "OPC Pvt Ltd", "KK"]
CITY_V2 = ["Mumbai 400001 IN", "Bengaluru 560001 IN", "Pune 411001 IN", "Chennai 600002 IN",
           "Hyderabad 500081 IN", "Kolkata 700001 IN", "Ahmedabad 380009 IN", "Gurugram 122002 IN",
           "Frankfurt 60311 DE", "Munich 80331 DE", "London EC2V 7HN GB", "Manchester M1 1AE GB",
           "New York NY 10005 US", "Chicago IL 60603 US", "Paris 75009 FR", "Lyon 69002 FR",
           "Singapore 048583 SG", "Hong Kong HK", "Dubai AE", "Zurich 8001 CH", "Tokyo 100-0005 JP",
           "Amsterdam 1012 NL", "Toronto ON M5H 2N2 CA", "Sydney NSW 2000 AU", "Madrid 28013 ES"]
STREET_V2 = ["12 MG Road", "4th Floor, Nariman Point", "221 Park Street", "Plot 7, MIDC", "Taunusanlage 12",
             "1 Churchill Place", "270 Park Avenue", "16 Bd des Italiens", "8 Marina Blvd",
             "Unit 5B, Hi-Tech Park", "Flat 302, Sai Residency", "No. 45, 2nd Cross, Indiranagar",
             "Tower B, Cyber City", "Shop 9, Linking Road", "77 Bishopsgate", "Calle de Alcalá 48",
             "Herengracht 210", "Suite 1200, 100 King St W", "Level 20, 1 Martin Place",
             "Gala 14, Andheri East", "SCO 112, Sector 17"]
REMIT_V2 = ["INV {n} consulting fees", "Invoice {n} / spare parts", "PO {n} advance payment", "Salary {mon} 2026",
            "Rent {mon} 2026 office", "Refund order {n}", "Freight charges BL {n}", "Licence fee Q{q} 2026",
            "Settlement of INV {n} and INV {n2}", "Tuition fees semester {q}", "Dividend payout {mon}",
            "Software subscription {n}", "AMC renewal {n}", "Import duty BOE {n}", "Royalty Q{q} 2026",
            "Retainer {mon} 2026", "Milestone {q} payment PO {n}", "GST refund ref {n}", "Loan EMI {mon} 2026",
            "Security deposit flat {n}", "Commission {mon} 2026", "Container demurrage {n}"]

# label synonyms per field (none of the OOD-only labels; see tests/test_v2_heldout.py)
L = {
    "ref": ["Our reference", "Our ref", "Reference", "Ref no", "Ref #", "Sender's reference", "Payment ref",
            "Transaction ref", "TRN", "Instruction ref", "Your ref (pls quote)", "Client ref", "Ref."],
    "amount": ["Amount", "Amt", "Amount to pay", "Remit amount", "Payment amount", "Total to remit",
               "Instructed amount", "Amount (ccy)", "Net amount"],
    "date": ["Value date", "Val dt", "VD", "Payment date", "Value dt", "Credit on", "Funds on", "Pay on",
             "Date of payment", "Requested value date", "Process on"],
    "oc": ["Ordering customer", "Remitter", "Debit account", "From", "Sender", "Ordering party",
           "Account holder (debit)", "Pay from", "Remitting party", "Applicant"],
    "oi": ["Our bank", "Ordering bank", "Remitter bank SWIFT", "Sender's bank BIC", "Debit bank", "Bank (debit side)",
           "Ordering institution", "Ord inst (52A)", "From bank", "Remitting bank SWIFT code"],
    "bn": ["Beneficiary", "Bene", "Pay to", "Beneficiary name & a/c", "Credit to", "To", "Bene name",
           "Beneficiary details", "In favour of", "Receiver"],
    "awi": ["Beneficiary bank", "Bene bank SWIFT", "Beneficiary bank BIC", "Account with institution",
            "AWI (57A)", "Credit bank", "Bene bank SWIFT code", "Beneficiary's bank", "To bank"],
    "remit": ["Purpose", "Details", "Remittance info", "Narration", "Info for beneficiary",
              "Description", "Reason for payment", "Invoice details", "Particulars", "Remarks to bene"],
    "charges": ["Charges", "Chrgs", "Charge type", "Bank charges", "Charges borne by", "Charge code", "Charges paid by",
                "Who pays charges", "71A charges"],
    "op": ["Op code", "23B", "Operation", "Operation code", "Type", "Op. code"],
}
ML = {  # multilingual-ish labels (German, French, Spanish, Hindi in Latin script)
    "ref": ["Referenz", "Référence", "Referencia", "Sandarbh sankhya"],
    "amount": ["Betrag", "Montant", "Importe", "Rakam"],
    "date": ["Valutadatum", "Date de valeur", "Fecha valor", "Tareekh"],
    "oc": ["Auftraggeber", "Donneur d'ordre", "Ordenante", "Bhejne wala"],
    "oi": ["Bank des Auftraggebers", "Banque émettrice", "Banco ordenante", "Bhejne wale ka bank"],
    "bn": ["Empfänger", "Bénéficiaire", "Beneficiario", "Paane wala"],
    "awi": ["Empfängerbank", "Banque du bénéficiaire", "Banco beneficiario", "Paane wale ka bank"],
    "remit": ["Verwendungszweck", "Motif", "Concepto", "Uddeshya"],
    "charges": ["Gebühren", "Frais", "Gastos", "Shulk"],
    "op": ["Op code", "Code opération", "Código", "Op code"],
}
ML_INTRO = ["Bitte folgende Zahlung ausführen:", "Merci d'exécuter le virement suivant :",
            "Por favor procesar la siguiente transferencia:", "Kripya yeh payment aaj kar dijiye:",
            "bhai ye payment jaldi kar do pls", "Zahlungsauftrag / payment order:"]
CHARGE_WORDS_V2 = {"OUR": ["OUR", "our", "ours", "OUR (we pay all charges)", "all charges on us", "remitter pays all",
                           "charges for our account", "we bear all charges"],
                   "SHA": ["SHA", "sha", "shared", "SHA - shared", "split charges (SHA)", "shared equally",
                           "each side pays its own bank"],
                   "BEN": ["BEN", "ben", "beneficiary pays", "BEN (charges to beneficiary)", "deduct from beneficiary",
                           "all charges to beneficiary"]}
GREET = ["Hi team,", "Dear Ops,", "Hello,", "Good morning,", "Hi", "Dear Sir/Madam,", "Hi Treasury team,",
         "Team,", "Dear Payments desk,", "Namaste,", "Hey all,", "Good afternoon team,"]
ASK = ["{please} process the below {payment} today.", "Kindly arrange the following transfer.",
       "Requesting an outward remittance as per details below.", "Can you release this wire asap?",
       "Please debit our account and remit as follows.", "Need this one out before cut-off pls.",
       "Below are the {payment} details, kindly execute.", "Find the transfer details below:"]
SIGN = ["Thanks,", "Regards,", "Thx", "Best,", "Warm regards,", "Cheers,", "Thanks & regards,", "Sincerely,"]
DISCLAIMER = ["This e-mail is confidential and may be privileged. If you are not the intended recipient, delete it.",
              "Please consider the environment before printing this email.",
              "Sent from my iPhone", "Sent from Outlook for Android",
              "CONFIDENTIALITY NOTICE: the information in this message is intended for the addressee only."]
BULLETS = ["", "", "- ", "* ", "• "]  # never "> ": that marks quoted (old) text in thread2


class GenV2(Gen):
    """Same target logic as v1; new surface formats, renderers, noise and distractors."""

    # ---------- primitives ----------
    def company(self) -> str:
        r = self.r
        return f"{r.choice(CO_A_V2)} {r.choice(CO_B_V2)} {r.choice(CO_C_V2)}"

    def person(self) -> str:
        return f"{self.r.choice(FIRST_V2)} {self.r.choice(LAST_V2)}"

    def account(self) -> str:
        r = self.r
        k = r.random()
        if k < 0.8:
            return super().account()
        if k < 0.9:
            return "".join(r.choice("0123456789") for _ in range(r.choice([14, 16])))  # Indian savings / current
        return f"{r.choice(['SB', 'CA', 'OD'])}-{r.randint(10 ** 9, 10 ** 10 - 1)}"

    def party(self, need_account: bool = True) -> dict:
        r = self.r
        name = self.company() if r.random() < 0.65 else self.person()
        acct = self.account() if (need_account or r.random() < 0.7) else None
        addr = f"{r.choice(STREET_V2)}, {r.choice(CITY_V2)}" if r.random() < 0.55 else None
        return {"account": acct, "name": name, "address": addr}

    def ref(self) -> str:
        r = self.r
        k = r.random()
        if k < 0.6:
            return super().ref()
        if k < 0.7:
            return f"RTGS{r.randint(10 ** 8, 10 ** 9 - 1)}"
        if k < 0.8:
            return f"WIRE-{r.randint(100000, 999999)}"
        if k < 0.9:
            return f"{r.choice(['ACME', 'HLX', 'ZEN', 'ORN'])}/PO/{r.randint(1, 9999):04d}"
        return f"TT{r.randint(26, 27)}{r.randint(1, 12):02d}{r.randint(1000, 9999)}"

    def remit(self) -> str:
        r = self.r
        return r.choice(REMIT_V2).format(n=f"{r.randint(1, 9999):04d}", n2=f"{r.randint(1, 9999):04d}",
                                         mon=r.choice(MONTHS), q=r.randint(1, 4))

    def amount(self, ccy: str) -> float:
        lo, hi = CCY_V2[ccy][1]
        r = self.r
        x = round(r.uniform(lo, hi), 2)
        if ccy == "JPY":
            x = float(int(x))
        k = r.random()
        if k < 0.2:
            x = float(round(x, -2)) or 100.0
        elif k < 0.27 and ccy == "INR":
            x = float(r.randint(1, 400) * 25000)  # round lakh amounts, so "x lakh" renders exactly
        return x

    def fmt_amount(self, x: float, ccy: str) -> str:
        """Every format renders the exact value (v1's f"{x:g}" was lossy above 6 digits)."""
        r = self.r
        k = r.random()
        cents = x != int(x)
        if ccy == "INR":
            if k < 0.15 and x >= 100000 and abs(x / 100000 - round(x / 100000, 2)) < 1e-9:
                v = round(x / 100000, 2)
                return f"{v:g} {r.choice(['lakh', 'lakhs', 'lac'])}"
            if k < 0.2 and x >= 10 ** 7 and abs(x / 10 ** 7 - round(x / 10 ** 7, 2)) < 1e-9:
                return f"{round(x / 10 ** 7, 2):g} crore"
            if k < 0.55:
                s = indian_format(x, decimals=cents or r.random() < 0.5)
                return s + "/-" if r.random() < 0.25 else s
        if ccy in ("EUR", "SEK") and k < 0.3:
            return f"{x:,.2f}".replace(",", "_").replace(".", ",").replace("_", ".")
        if ccy == "CHF" and k < 0.3:
            return f"{x:,.2f}".replace(",", "'")
        if k < 0.6:
            return f"{x:,.2f}" if cents or r.random() < 0.6 else f"{x:,.0f}"
        if k < 0.8:
            return f"{x:.2f}"
        if k < 0.9:
            return f"{x:,.2f}".replace(",", " ")
        return f"{x:,.2f}".replace(",", " ").replace(".", ",")  # 1 234 567,89

    def fmt_ccy(self, ccy: str) -> str:
        r = self.r
        if ccy in CCY_SYMBOL and r.random() < 0.35:
            return r.choice(CCY_SYMBOL[ccy])
        return ccy if r.random() < 0.85 else ccy.lower()

    def fmt_date(self, d: dt.date) -> str:
        r = self.r
        if r.random() < 0.4:
            return super().fmt_date(d)
        return r.choice([
            f"{d.day:02d} {MONTHS_LONG[d.month - 1]} {d.year}",
            f"{DAYS_LONG[d.weekday()]}, {d.day} {MONTHS_LONG[d.month - 1]} {d.year}",
            f"{MONTHS[d.month - 1]} {d.day}, {d.year}",
            f"{d.day}/{d.month}/{d.year}",
            d.strftime("%d/%m/%y"),
            d.strftime("%d.%m.%y"),
            f"{d.day}-{MONTHS[d.month - 1]}-{d.year}",
            f"{d.day} {MONTHS_LONG[d.month - 1].lower()} {d.year}",
        ])

    def bad_date(self) -> str:
        r = self.r
        if r.random() < 0.5:
            return super().bad_date()
        return r.choice(["31/06/2026", "2026-04-31", "29 Feb 2026", "31-04-2026", "00/12/2026", "2026-02-29",
                         "31.11.2026", "April 31, 2026", "30/02/2027", "2027-13-01"])

    def w2(self, key: str, ml: bool = False) -> str:
        """A label for a field: synonym + typo/case noise."""
        lab = self.r.choice(ML[key] if ml else L[key])
        return self.noise(lab)

    def noise(self, s: str, p: float = 0.18) -> str:
        """Typing noise: transposed / dropped / doubled letter, or random case. (No OCR letter-digit swaps.)"""
        r = self.r
        if r.random() < p:
            idx = [i for i, c in enumerate(s) if c.isalpha()]
            if len(idx) >= 4:
                i = r.choice(idx[1:-1])
                k = r.random()
                if k < 0.4 and s[i + 1].isalpha():
                    s = s[:i] + s[i + 1] + s[i] + s[i + 2:]
                elif k < 0.7:
                    s = s[:i] + s[i + 1:]
                else:
                    s = s[:i] + s[i] + s[i:]
        k = r.random()
        return s.upper() if k < 0.1 else s.lower() if k < 0.2 else s

    def distractor(self, f) -> str:
        r = self.r
        d2 = self.date()
        ccy = f["currency"] or "USD"
        amt2 = self.fmt_amount(self.amount(ccy if ccy in CCY_V2 else "USD"), ccy if ccy in CCY_V2 else "USD")
        opts = [
            f"Invoice total incl. GST: {ccy} {amt2} (we pay part now)",
            f"Bank charges estimate approx {ccy} {r.randint(5, 60)}.00",
            f"Ticket #{r.randint(10000, 99999)} raised on {self.fmt_date(d2)}",
            f"Call me at +44 20 {r.randint(1000, 9999)} {r.randint(1000, 9999)} if needed",
            f"Old instruction {self.ref()} for {ccy} {amt2} stands cancelled",
            f"Our VAT no DE{r.randint(10 ** 8, 10 ** 9 - 1)}",
            f"PAN on file, GSTIN 27AAAC{r.randint(1000, 9999)}F1Z{r.randint(1, 9)}",
            f"Cut-off today is {r.randint(1, 4)}:30 pm.",
            f"Last payment to them went out on {self.fmt_date(d2)}.",
            f"(Supersedes draft ref {self.ref()})",
        ]
        if f["account_with_institution"] in INDIAN_BANKS or f["ordering_institution"] in INDIAN_BANKS:
            opts.append(f"IFSC: {r.choice(['HDFC', 'SBIN', 'ICIC', 'UTIB'])}0{r.randint(100000, 999999)}")
        return r.choice(opts)

    # ---------- one example ----------
    def example(self, idx: int) -> dict:
        r = self.r
        ccy = r.choices(list(CCY_V2), weights=[v[0] for v in CCY_V2.values()])[0]
        amt = self.amount(ccy)
        d = self.date()
        f = {
            "sender_reference": self.ref(),
            "bank_operation_code": "CRED" if r.random() < 0.85 else r.choice(["SPAY", "SPRI", "SSTD", "CRTS"]),
            "value_date": d.isoformat(), "currency": ccy, "amount": amt,
            "ordering_customer": self.party(), "ordering_institution": r.choice(BANKS_V2),
            "account_with_institution": r.choice(BANKS_V2), "beneficiary": self.party(),
            "remittance_info": self.remit(), "charges": r.choice(["OUR", "SHA", "SHA", "BEN"]),
        }
        surf = {"date": self.fmt_date(d), "ccy": self.fmt_ccy(ccy), "amt": self.fmt_amount(amt, ccy),
                "charges": r.choice(CHARGE_WORDS_V2[f["charges"]])}
        # optional fields absent from the text (independently, so "only 52A" happens as often as "only 57A")
        for k, p in (("bank_operation_code", 0.45), ("ordering_institution", 0.35),
                     ("account_with_institution", 0.35), ("remittance_info", 0.2)):
            if r.random() < p:
                f[k] = None
        errors: list[str] = []
        if r.random() < 0.30:
            kind = r.choices(["ccy", "date", "bic", "amt", "charges", "missing"], weights=[1, 1, 1, 1, 1, 1.4])[0]
            if kind == "ccy":
                bad = r.choice(BAD_CCY_V2); assert bad not in ISO4217
                surf["ccy"] = bad; f["currency"] = None
                errors.append(f"INVALID_CURRENCY: currency: {bad} is not an ISO-4217 code")
            elif kind == "date":
                surf["date"] = self.bad_date(); f["value_date"] = None
                errors.append(f"INVALID_DATE: value_date: {surf['date']} is not a real calendar date")
            elif kind == "bic":
                which = r.choice(["ordering_institution", "account_with_institution"])
                bad = r.choice(BAD_BICS_V2); f[which] = None
                surf["badbic"] = (which, bad)
                errors.append(f"INVALID_BIC: {which}: {bad} is not a valid BIC")
            elif kind == "amt":
                bad = r.choice(["0.00", "-" + self.fmt_amount(amt, ccy), "0", "-" + f"{amt:.2f}",
                                "−" + f"{amt:,.2f}", "-0.00"])
                surf["amt"] = bad; f["amount"] = None
                errors.append(f"NON_POSITIVE_AMOUNT: amount: {bad} is not a positive amount")
            elif kind == "charges":
                bad = r.choice(BAD_CHARGES_V2); surf["charges"] = bad; f["charges"] = None
                errors.append(f"INVALID_CHARGES: charges: {bad} is not OUR/SHA/BEN")
            else:
                miss = r.choice(["charges", "beneficiary", "value_date"])
                f[miss] = None; surf["missing"] = miss
                errors.append(f"MISSING_REQUIRED: {miss}: not stated in the text")
        text = self.render(f, surf)
        target = {"fields": f, "errors": errors}
        Extraction.model_validate(target)
        return {"id": f"v2-{idx:05d}", "synthetic": True, "style": self._style, "text": text, "target": target}

    # ---------- renderers ----------
    STYLES = {"email2": 14, "chat2": 10, "swift2": 12, "form2": 9, "sections2": 7, "csv2": 5, "yaml2": 5,
              "letter2": 7, "multi2": 7, "thread2": 8, "v1": 16}

    def render(self, f, surf) -> str:
        style = self.r.choices(list(self.STYLES), weights=list(self.STYLES.values()))[0]
        if style == "v1" and surf.get("charges") in set(BAD_CHARGES_V2) - set(BAD_CHARGES):
            style = "email2"  # v1's SWIFT renderer only knows v1's bad charge words
        self._style = style
        if style == "v1":
            s = super().render(f, surf)
            if self.r.random() < 0.3:
                s = s + "\n\n" + self.distractor(f)
            return s
        return getattr(self, f"r_{style}")(f, surf).strip()

    def _has_bic(self, f, s, key):
        return bool(f[key]) or s.get("badbic", ("",))[0] == key

    def _money2(self, s):
        c, a = s["ccy"], s["amt"]
        if c in ("$", "US$", "€", "£", "₹", "¥", "Rs.", "Rs"):
            return f"{c}{a}" if self.r.random() < 0.5 else f"{c} {a}"
        return f"{c} {a}" if self.r.random() < 0.7 else f"{a} {c}"

    def _party_txt(self, p, sep=", "):
        r = self.r
        bits = []
        if p.get("account"):
            bits.append(p["account"] if r.random() < 0.4 else f"{r.choice(['a/c', 'A/C', 'acct', 'account', 'Acc no', 'IBAN' if p['account'][:2].isalpha() else 'a/c no'])} {p['account']}")
        bits.append(p["name"])
        if p.get("address"):
            bits.append(p["address"])
        if r.random() < 0.3 and len(bits) > 1:  # name first, then account
            bits[0], bits[1] = bits[1], bits[0]
        return sep.join(bits)

    def _items(self, f, s, ml=False):
        """(field key, label, value) for every field present in the text, in random order."""
        it = [("ref", f["sender_reference"]), ("amount", self._money2(s))]
        if s.get("missing") != "value_date":
            it.append(("date", s["date"]))
        it.append(("oc", self._party_txt(f["ordering_customer"])))
        if self._has_bic(f, s, "ordering_institution"):
            it.append(("oi", self.bic_s(f, s, "ordering_institution")))
        if s.get("missing") != "beneficiary":
            it.append(("bn", self._party_txt(f["beneficiary"])))
        if self._has_bic(f, s, "account_with_institution"):
            it.append(("awi", self.bic_s(f, s, "account_with_institution")))
        if f["remittance_info"]:
            it.append(("remit", f["remittance_info"]))
        if s.get("missing") != "charges":
            it.append(("charges", s["charges"]))
        if f["bank_operation_code"]:
            it.append(("op", f["bank_operation_code"]))
        self.r.shuffle(it)
        return [(k, self.w2(k, ml), v) for k, v in it]

    def r_email2(self, f, s):
        r = self.r
        sep = r.choice([": ", " : ", " - ", ":  ", " – "])
        b = r.choice(BULLETS)
        lines = [r.choice(GREET), "", r.choice(ASK).format(please=self.w("please").capitalize(), payment=self.w("payment"))]
        if r.random() < 0.3:
            lines.append("")
        lines += [f"{b}{lab}{sep}{v}" for _, lab, v in self._items(f, s)]
        if r.random() < 0.5:
            lines += ["", self.distractor(f)]
        lines += ["", r.choice(SIGN), self.person()]
        if r.random() < 0.4:
            lines.append(r.choice(["Treasury Ops", "Accounts Payable", "Finance team", f"Ext. {r.randint(100, 999)}"]))
        if r.random() < 0.3:
            lines += ["", r.choice(DISCLAIMER)]
        return "\n".join(lines)

    def r_chat2(self, f, s):
        r = self.r
        who = [self.r.choice(FIRST_V2), self.r.choice(FIRST_V2)]
        h, m = r.randint(8, 18), r.randint(0, 50)
        msgs = []
        chunks = []
        for k, lab, v in self._items(f, s):
            if k in ("amount", "bn") and r.random() < 0.5:
                chunks.append(("send " if k == "amount" else "to ") + v)
            elif r.random() < 0.5:
                chunks.append(f"{lab.lower()} {v}")
            else:
                chunks.append(f"{lab} {v}")
        opener = r.choice(["pls do a payment", "urgent pymt", "need one wire today", "ek payment karna hai",
                           "quick one", "can u release a transfer"])
        msgs.append(opener)
        i = 0
        while i < len(chunks):
            n = r.randint(1, 3)
            msgs.append(", ".join(chunks[i:i + n]))
            i += n
        if r.random() < 0.4:
            msgs.insert(r.randint(1, len(msgs)), self.distractor(f))
        out = []
        for j, mtxt in enumerate(msgs):
            m += r.randint(0, 3)
            stamp = r.choice([f"[{h:02d}:{m % 60:02d}] {who[0]}: ", f"{who[0]} {h:02d}:{m % 60:02d} - ", ""])
            out.append(stamp + mtxt)
        if r.random() < 0.5:
            out.append(r.choice([f"[{h:02d}:{(m + 2) % 60:02d}] {who[1]}: ok will do", f"{who[1]}: noted 👍",
                                 f"{who[1]}: done?"]))
        return "\n".join(out)

    def r_swift2(self, f, s):
        """Standard tag ORDER (never shuffled), upper-case tags; extra and distractor tags vary positions."""
        r = self.r
        oc, bn = f["ordering_customer"], f["beneficiary"]
        ccy = s["ccy"] if len(s["ccy"]) == 3 and s["ccy"].isalpha() else (f["currency"] or s["ccy"])
        ccy = ccy.upper()
        amt = s["amt"]
        if not amt.startswith(("-", "−")) and not any(w in amt for w in ("lakh", "lac", "crore")) and f["amount"]:
            amt = f"{f['amount']:.2f}".replace(".", ",")
        date = s["date"]
        if f["value_date"] and r.random() < 0.75:
            date = dt.date.fromisoformat(f["value_date"]).strftime("%y%m%d")
        out = []
        if r.random() < 0.4 and f["ordering_institution"] and f["account_with_institution"]:
            out.append(f"{{1:F01{f['ordering_institution'][:8]}AXXX{r.randint(10 ** 9, 10 ** 10 - 1)}}}"
                       f"{{2:I103{f['account_with_institution'][:8]}XN}}{{3:{{108:{r.randint(1000, 9999)}}}}}{{4:")
        out.append(f":20:{f['sender_reference']}")
        if r.random() < 0.15:
            out.append(f":13C:/SNDTIME/{r.randint(1000, 2359):04d}+0530")
        if f["bank_operation_code"]:
            out.append(f":23B:{f['bank_operation_code']}")
        if s.get("missing") != "value_date":
            out.append(f":32A:{date}{ccy}{amt}")
        else:
            out.append(f":32A:{ccy}{amt}")
        if r.random() < 0.2 and f["amount"] and f["currency"]:
            out.append(f":33B:{ccy}{amt}")  # instructed amount = same amount (common in real MT103s)
        out.append(":50K:" + "\n".join(x for x in [("/" + oc["account"]) if oc["account"] else None, oc["name"], oc["address"]] if x))
        if self._has_bic(f, s, "ordering_institution"):
            out.append(f":52A:{self.bic_s(f, s, 'ordering_institution')}")
        if r.random() < 0.15:
            out.append(f":53A:{r.choice(BANKS_V2)}")  # sender's correspondent: NOT 52A/57A
        if r.random() < 0.15:
            out.append(f":56A:{r.choice(BANKS_V2)}")  # intermediary: NOT 52A/57A
        if self._has_bic(f, s, "account_with_institution"):
            out.append(f":57A:{self.bic_s(f, s, 'account_with_institution')}")
        if s.get("missing") != "beneficiary":
            out.append(":59:" + "\n".join(x for x in [("/" + bn["account"]) if bn["account"] else None, bn["name"], bn["address"]] if x))
        if f["remittance_info"]:
            out.append(f":70:{f['remittance_info']}")
        if s.get("missing") != "charges":
            ch = s["charges"] if s["charges"] in BAD_CHARGES_V2 else f["charges"]
            out.append(f":71A:{ch}")
        if r.random() < 0.2:
            out.append(f":71F:{ccy}{r.randint(5, 40)},00")  # sender's charges: a distractor amount
        if r.random() < 0.2:
            out.append(f":72:/ACC/{r.choice(['PLS CREDIT SAME DAY', 'URGENT', 'TEL BEFORE PAYMENT'])}")
        if out[0].startswith("{1:"):
            out.append("-}")
        wrap = r.random()
        if wrap < 0.3:
            return "\n".join([r.choice(GREET), "", r.choice(["MT103 below, pls check before release:",
                                                              "Copy of the swift we got:", "Draft MT103:"]), ""] + out
                             + ["", r.choice(SIGN), self.person()])
        if wrap < 0.45:
            return "\n".join(out) + "\n\n" + self.distractor(f)
        return "\n".join(out)

    def r_form2(self, f, s):
        r = self.r
        items = self._items(f, s)
        width = max(len(lab) for _, lab, _ in items) + r.randint(2, 6)
        fill = r.choice([" ", ".", "_"])
        title = r.choice(["PAYMENT INSTRUCTION FORM", "Outward Remittance Request", "WIRE TRANSFER REQUEST",
                          "Fund Transfer Form (Treasury)", "Payment Request #" + str(r.randint(1000, 9999))])
        lines = [title, r.choice(["=" * len(title), "-" * len(title), ""])]
        for _, lab, v in items:
            lines.append(f"{lab.ljust(width, fill)} {v}" if fill != " " else f"{lab}:".ljust(width + 1) + f" {v}")
        if r.random() < 0.4:
            lines += ["", f"Authorised by: {self.person()}", f"Checked by: {self.person()}"]
        return "\n".join(lines)

    def r_sections2(self, f, s):
        r = self.r
        oc, bn = f["ordering_customer"], f["beneficiary"]
        hdr = r.choice([lambda t: f"[{t}]", lambda t: f"## {t}", lambda t: f"{t.upper()}:", lambda t: f"--- {t} ---"])

        def block(title, p, bic_key, bic_label):
            out = [hdr(title), f"{r.choice(['Name', 'Name', 'Account name'])}: {p['name']}"]
            if p.get("account"):
                out.append(f"{r.choice(['Account', 'A/c no', 'Account number', 'IBAN / a/c'])}: {p['account']}")
            if p.get("address"):
                out.append(f"{r.choice(['Address', 'Addr'])}: {p['address']}")
            if self._has_bic(f, s, bic_key):
                out.append(f"{bic_label}: {self.bic_s(f, s, bic_key)}")
            return out

        secs = [block(r.choice(["Ordering customer", "Remitter", "Debit side", "From"]), oc, "ordering_institution",
                      r.choice(["Bank SWIFT", "BIC", "Bank BIC"]))]
        if s.get("missing") != "beneficiary":
            secs.append(block(r.choice(["Beneficiary", "Credit side", "Bene", "To"]), bn, "account_with_institution",
                              r.choice(["Bank SWIFT", "BIC", "Bank BIC"])))
        elif self._has_bic(f, s, "account_with_institution"):
            secs.append([hdr("Beneficiary bank"), f"BIC: {self.bic_s(f, s, 'account_with_institution')}"])
        pay = [hdr(r.choice(["Payment", "Transfer", "Details"])), f"{self.w2('ref')}: {f['sender_reference']}",
               f"{self.w2('amount')}: {self._money2(s)}"]
        if s.get("missing") != "value_date":
            pay.append(f"{self.w2('date')}: {s['date']}")
        if f["remittance_info"]:
            pay.append(f"{self.w2('remit')}: {f['remittance_info']}")
        if s.get("missing") != "charges":
            pay.append(f"{self.w2('charges')}: {s['charges']}")
        if f["bank_operation_code"]:
            pay.append(f"{self.w2('op')}: {f['bank_operation_code']}")
        secs.insert(r.randint(0, len(secs)), pay)
        return "\n\n".join("\n".join(x) for x in secs)

    def r_csv2(self, f, s):
        r = self.r
        sep = r.choice([",", "\t", ";"])
        items = self._items(f, s)
        q = lambda v: f'"{v}"' if (sep in v or "," in v or r.random() < 0.3) else v  # noqa: E731
        head = sep.join(q(lab) for _, lab, _ in items)
        row = sep.join(q(v) for _, _, v in items)
        pre = r.choice(["", "Exported from ERP, 1 row:", "payments_batch.csv (row 1 of 1)", "Upload file:"])
        return "\n".join(x for x in [pre, head, row] if x)

    def r_yaml2(self, f, s):
        r = self.r
        ind = r.choice(["  ", "    "])
        out = [r.choice(["payment:", "transfer:", "instruction:"])]
        for k, lab, v in self._items(f, s):
            key = lab.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("'", "")
            if k in ("oc", "bn"):
                p = f["ordering_customer"] if k == "oc" else f["beneficiary"]
                out.append(f"{ind}{key}:")
                for pk in ("name", "account", "address"):
                    if p.get(pk):
                        out.append(f"{ind * 2}{pk}: {p[pk]}")
            else:
                out.append(f"{ind}{key}: {v}")
        return "\n".join(out)

    def r_letter2(self, f, s):
        r = self.r
        oc, bn = f["ordering_customer"], f["beneficiary"]
        sents = [f"We, {oc['name']}" + (f" (account {oc['account']})" if oc.get("account") else "")
                 + ", hereby request you to remit the amount below."]
        sents.append(f"The sum is {self._money2(s)}"
                     + (f", payable to {self._party_txt(bn)}" if s.get("missing") != "beneficiary" else "") + ".")
        if s.get("missing") != "value_date":
            sents.append(r.choice(["Value date requested: ", "Kindly credit the funds with value ",
                                   "The payment should be effected on "]) + s["date"] + ".")
        sents.append(r.choice(["Please mention reference ", "Our internal reference is ", "Kindly quote "])
                     + f["sender_reference"] + (" in all correspondence." if r.random() < 0.5 else "."))
        if self._has_bic(f, s, "ordering_institution"):
            sents.append(f"Please debit our account held with your branch, BIC {self.bic_s(f, s, 'ordering_institution')}.")
        if self._has_bic(f, s, "account_with_institution"):
            sents.append(f"The beneficiary banks with SWIFT {self.bic_s(f, s, 'account_with_institution')}.")
        if f["remittance_info"]:
            sents.append(f"This payment is towards {f['remittance_info']}.")
        if s.get("missing") != "charges":
            sents.append(f"Charges: {s['charges']}.")
        if f["bank_operation_code"]:
            sents.append(f"Bank operation code {f['bank_operation_code']}.")
        head, rest = sents[:1], sents[1:]
        r.shuffle(rest)
        if r.random() < 0.4:
            rest.insert(r.randint(0, len(rest)), self.distractor(f) + ".")
        body = " ".join(head + rest)
        return "\n".join([r.choice(["To,\nThe Branch Manager", "Dear Sir/Madam,", "To the Payments Department,"]), "",
                          r.choice(["Sub: Request for outward remittance", "Re: wire transfer", "Subject: Payment"]), "",
                          body, "", r.choice(["Yours faithfully,", "Yours sincerely,", "Regards,"]),
                          f"For {oc['name']}", "(Authorised Signatory)"])

    def r_multi2(self, f, s):
        r = self.r
        sep = r.choice([": ", " : ", " - "])
        lines = [r.choice(ML_INTRO), ""]
        for _, lab, v in self._items(f, s, ml=r.random() < 0.8):
            lines.append(f"{lab}{sep}{v}")
        lines += ["", r.choice(["Danke", "Merci", "Gracias", "Dhanyavaad", "Thanks / Danke", "Shukriya"])]
        return "\n".join(lines)

    def r_thread2(self, f, s):
        r = self.r
        sent = self.date()
        hdr = [r.choice(["-----Original Message-----", "---------- Forwarded message ---------",
                         "Begin forwarded message:", "________________________________"]),
               f"From: {self.person()} <{r.choice(['ops', 'treasury', 'ap', 'finance'])}{r.randint(1, 99)}@example.com>",
               f"Sent: {DAYS_LONG[sent.weekday()][:3]}, {sent.day:02d} {MONTHS[sent.month - 1]} {sent.year} "
               f"{r.randint(8, 19):02d}:{r.randint(0, 59):02d}",
               f"Subject: {r.choice(['RE: RE: payment', 'FW: wire instructions', 'Fwd: urgent remittance', 'RE: pymt req – revised'])}",
               ""]
        new_part = self.r_email2(f, s) if r.random() < 0.6 else self.r_chat2(f, s)
        old_ccy = f["currency"] or "USD"
        old_amt = self.fmt_amount(self.amount(old_ccy if old_ccy in CCY_V2 else "USD"), old_ccy if old_ccy in CCY_V2 else "USD")
        old = [f"> {r.choice(['Earlier draft (do NOT use)', 'Previous instruction (cancelled)', 'Older mail below, ignore'])}:",
               f"> {self.w2('ref')}: {self.ref()}",
               f"> {self.w2('amount')}: {old_ccy} {old_amt}",
               f"> {self.w2('date')}: {self.fmt_date(self.date())}"]
        top = [r.choice(["Pls action the latest one below.", "Revised details below, thanks.", "FYI - see below",
                         "Forwarding for processing."]), ""]
        parts = top + hdr + [new_part, ""] + (old if r.random() < 0.7 else [])
        if r.random() < 0.3:
            parts += ["", r.choice(DISCLAIMER)]
        return "\n".join(parts)


def generate_v2(n: int, seed: int, exclude_texts: frozenset = frozenset()) -> list[dict]:
    g = GenV2(seed)
    rows, seen = [], set(exclude_texts)
    while len(rows) < n:
        ex = g.example(len(rows))
        h = hashlib.sha1(ex["text"].encode()).hexdigest()
        if h in seen:
            continue
        seen.add(h)
        rows.append(ex)
    return rows


def ood_text_hashes() -> frozenset:
    p = Path(__file__).parent / "ood_test.jsonl"
    if not p.exists():
        return frozenset()
    return frozenset(hashlib.sha1(json.loads(x)["text"].encode()).hexdigest()
                     for x in open(p, encoding="utf-8") if x.strip())


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--n", type=int, default=6000)
    ap.add_argument("--seed", type=int, default=2026)
    ap.add_argument("--out", default=str(Path(__file__).parent / "v2"))
    a = ap.parse_args()
    rows = generate_v2(a.n, a.seed, ood_text_hashes())
    tr, va, te = split_ids(len(rows), a.seed)
    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    counts = {}
    for name, idx in (("train", tr), ("val", va), ("test", te)):
        with open(out / f"{name}.jsonl", "w", encoding="utf-8") as fh:
            for i in sorted(idx):
                fh.write(json.dumps(rows[i], ensure_ascii=False) + "\n")
        counts[name] = len(idx)
    styles, codes = {}, {}
    for row in rows:
        styles[row["style"]] = styles.get(row["style"], 0) + 1
        for e in row["target"]["errors"]:
            codes[e.split(":")[0]] = codes.get(e.split(":")[0], 0) + 1
    manifest = {"synthetic": True, "note": "SYNTHETIC format-diverse training data (data/generate_v2.py); not real payments.",
                "seed": a.seed, "n": len(rows), "splits": counts, "styles": dict(sorted(styles.items())),
                "error_codes": dict(sorted(codes.items())),
                "examples_with_errors": sum(1 for r in rows if r["target"]["errors"])}
    (out / "MANIFEST.json").write_text(json.dumps(manifest, indent=2))
    print(json.dumps(manifest))


if __name__ == "__main__":
    assert all(c not in ISO4217 for c in BAD_CCY_V2)
    assert all(not BIC_RE.match(b) for b in BAD_BICS_V2)
    main()

In [ ]:
%%writefile mt103/data/generate_ood.py
"""Out-of-distribution (OOD) SYNTHETIC test set: different seed, renderers and noise patterns.

Everything produced here is synthetic. The goal is to check whether the in-distribution test score
generalises or only reflects memorised surface format. Compared with data/generate.py, this set uses:

  * a different seed (20261123) and NEW renderers, none of which appear in training:
      - "swift_shuffled": MT103 tag blocks in a random order, tags often lower-case (:32a:, :50k:, :71a:)
      - "narrative": one free-text paragraph, with the facts in a random order
      - "pipe_table": a `| key | value |` table, rows in random order, unseen key names
      - "kv_mixed": a `key=value; ...` one-liner in random order, unseen key names
  * OCR-style character swaps (rn->m, l->1, O->0, e->c ...) applied ONLY to labels and filler words,
    never to the values being extracted, so the gold target stays unambiguous
  * extra free-text remarks, including distractor numbers, dates and amounts that must be ignored
  * date formats not used in training (5th March 2026, 2026/03/05, 20260305, Thu 5 Mar 26, 5.3.2026,
    05-MAR-2026), plus distractor dates in other formats
  * impossible values from NEW lists (currencies, BICs, dates, charge words that training never saw)

Gold targets follow the same schema and rules as the main set and are validated with schema.Extraction.

Usage: python data/generate_ood.py --n 100 --seed 20261123 --out data/ood_test.jsonl
"""
from __future__ import annotations

import argparse
import datetime as dt
import hashlib
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from data.generate import BANKS, MONTHS, MONTHS_LONG, Gen  # noqa: E402
from schema import BIC_RE, ISO4217, Extraction  # noqa: E402

# Impossible values the training generator never produced (checked in tests/test_ood.py).
OOD_BAD_CCY = ["USDD", "EURO", "YEN", "RUP", "XXY", "GPB", "DLR", "INRS"]
OOD_BAD_BICS = ["ICIC0INBB", "CITI-US-33", "DEUT DE FF", "SCBL", "BOFAUS3NXXXX", "HDFCINB", "UBSW1234ZH"]
OOD_BAD_DATES = ["2026-02-30", "31st April 2026", "2026/06/31", "20260231", "0/05/2026", "31-SEP-2026",
                 "30th February 2027", "2026.11.31"]
OOD_BAD_CHARGES = ["SPLIT", "SHR", "BOTH", "OUR/BEN", "FREE"]
OOD_KINDS = ["ccy", "date", "bic", "amt", "charges", "missing"]

# OCR confusions, applied to labels/filler only.
OCR = [("rn", "m"), ("m", "rn"), ("l", "1"), ("O", "0"), ("e", "c"), ("i", "l"), ("S", "5"), ("cl", "d")]

REMARKS = [
    "Note: last month we sent {amt2} to them for a different invoice, ignore that one.",
    "Client called on {d2} to confirm, ticket closed after.",
    "Pls call me on +91 98{n7} if anything is unclear.",
    "Previous ref {ref2} was cancelled, do NOT reuse it.",
    "FYI compliance cleared this on {d2}.",
    "Remember the cut-off is 3pm IST; the {amt2} batch can wait.",
    "Attaching the invoice scan (page 2 of 3).",
    "This replaces the instruction from {d2}.",
]


def ood_fmt_date(r, d: dt.date) -> str:
    sfx = "th" if 11 <= d.day <= 13 else {1: "st", 2: "nd", 3: "rd"}.get(d.day % 10, "th")
    return r.choice([
        f"{d.day}{sfx} {MONTHS_LONG[d.month - 1]} {d.year}",
        d.strftime("%Y/%m/%d"),
        d.strftime("%Y%m%d"),
        f"{d.strftime('%a')} {d.day} {MONTHS[d.month - 1]} {d.strftime('%y')}",
        f"{d.day}.{d.month}.{d.year}",
        f"{d.day:02d}-{MONTHS[d.month - 1].upper()}-{d.year}",
        f"{MONTHS[d.month - 1]}. {d.day} {d.year}",
    ])


class OodGen(Gen):
    """Reuses the primitive value generators from Gen; replaces errors, dates and renderers."""

    def ocr(self, s: str, p: float = 0.35) -> str:
        if self.r.random() >= p:
            return s
        cands = [(a, b) for a, b in OCR if a in s]
        if not cands:
            return s
        a, b = self.r.choice(cands)
        i = s.find(a)
        return s[:i] + b + s[i + len(a):]

    def lab(self, s: str) -> str:
        """A label: OCR noise + random case."""
        s = self.ocr(s)
        k = self.r.random()
        return s.lower() if k < 0.25 else s.upper() if k < 0.4 else s

    def remark(self) -> str:
        r = self.r
        d2 = self.date()
        return r.choice(REMARKS).format(
            amt2=f"{r.choice(['USD', 'EUR', 'INR'])} {r.randint(100, 99999):,}.00",
            d2=r.choice([d2.strftime("%d/%m/%Y"), d2.isoformat(), f"{d2.day} {MONTHS[d2.month - 1]}"]),
            n7=f"{r.randint(0, 99999999):08d}", ref2=self.ref())

    def example(self, idx: int) -> dict:
        r = self.r
        ccy = r.choice(["USD", "EUR", "GBP", "INR", "SGD", "JPY", "CHF", "AED", "HKD"])
        amt = self.amount(ccy)
        d = self.date()
        f = {
            "sender_reference": self.ref(),
            "bank_operation_code": "CRED" if r.random() < 0.85 else r.choice(["SPAY", "SPRI", "SSTD", "CRTS"]),
            "value_date": d.isoformat(), "currency": ccy, "amount": amt,
            "ordering_customer": self.party(), "ordering_institution": r.choice(BANKS),
            "account_with_institution": r.choice(BANKS), "beneficiary": self.party(),
            "remittance_info": self.remit(), "charges": r.choice(["OUR", "SHA", "BEN"]),
        }
        surf = {"date": ood_fmt_date(r, d), "ccy": self.fmt_ccy(ccy), "amt": self.fmt_amount(amt, ccy),
                "charges": r.choice({"OUR": ["OUR", "our", "all charges ours"],
                                     "SHA": ["SHA", "sha", "shared"],
                                     "BEN": ["BEN", "ben", "beneficiary pays"]}[f["charges"]])}
        for k, p in (("bank_operation_code", 0.45), ("ordering_institution", 0.3),
                     ("account_with_institution", 0.3), ("remittance_info", 0.2)):
            if r.random() < p:
                f[k] = None
        errors: list[str] = []
        # Higher error rate than training (35% vs 22%) so the per-code analysis has enough support.
        if r.random() < 0.35:
            kind = OOD_KINDS[idx % len(OOD_KINDS)]  # round-robin: balanced error codes
            if kind == "ccy":
                bad = r.choice(OOD_BAD_CCY)
                surf["ccy"] = bad; f["currency"] = None
                errors.append(f"INVALID_CURRENCY: currency: {bad} is not an ISO-4217 code")
            elif kind == "date":
                surf["date"] = r.choice(OOD_BAD_DATES); f["value_date"] = None
                errors.append(f"INVALID_DATE: value_date: {surf['date']} is not a real calendar date")
            elif kind == "bic":
                which = r.choice(["ordering_institution", "account_with_institution"])
                bad = r.choice(OOD_BAD_BICS); f[which] = None
                surf["badbic"] = (which, bad)
                errors.append(f"INVALID_BIC: {which}: {bad} is not a valid BIC")
            elif kind == "amt":
                bad = r.choice(["(" + self.fmt_amount(amt, ccy) + ")", "0.0", "- " + f"{amt:.2f}", "nil"])
                if bad == "nil":
                    bad = "0"
                surf["amt"] = bad; f["amount"] = None
                errors.append(f"NON_POSITIVE_AMOUNT: amount: {bad} is not a positive amount")
            elif kind == "charges":
                bad = r.choice(OOD_BAD_CHARGES); surf["charges"] = bad; f["charges"] = None
                errors.append(f"INVALID_CHARGES: charges: {bad} is not OUR/SHA/BEN")
            else:
                miss = r.choice(["charges", "beneficiary", "value_date"])
                f[miss] = None; surf["missing"] = miss
                errors.append(f"MISSING_REQUIRED: {miss}: not stated in the text")
        style = r.choice(["swift_shuffled", "narrative", "pipe_table", "kv_mixed"])
        text = getattr(self, f"o_{style}")(f, surf).strip()
        target = {"fields": f, "errors": errors}
        Extraction.model_validate(target)
        return {"id": f"ood-{idx:04d}", "synthetic": True, "ood_style": style, "text": text, "target": target}

    # ---------- renderers (all new) ----------
    def _items(self, f, s):
        """(label_variants, value) items common to the table-like renderers; None values skipped."""
        oc, bn = f["ordering_customer"], f["beneficiary"]
        it = [(["Sender ref", "Our ref#", "Txn reference"], f["sender_reference"]),
              (["Amount", "Sum", "Transfer amount"], self.money(s))]
        if s.get("missing") != "value_date":
            it.append((["Settle on", "Value", "Execution date"], s["date"]))
        it.append((["Remitter", "Payer", "Debtor"], self.party_s(oc, " / ")))
        if f["ordering_institution"] or s.get("badbic", ("",))[0] == "ordering_institution":
            it.append((["Payer bank BIC", "Sending bank", "Debtor agent"], self.bic_s(f, s, "ordering_institution")))
        if s.get("missing") != "beneficiary":
            it.append((["Payee", "Creditor", "Recipient"], self.party_s(bn, " / ")))
        if f["account_with_institution"] or s.get("badbic", ("",))[0] == "account_with_institution":
            it.append((["Payee bank BIC", "Receiving bank", "Creditor agent"], self.bic_s(f, s, "account_with_institution")))
        if f["remittance_info"]:
            it.append((["Narrative", "Payment details", "Memo"], f["remittance_info"]))
        if s.get("missing") != "charges":
            it.append((["Fees", "Charge bearer", "Costs"], s["charges"]))
        if f["bank_operation_code"]:
            it.append((["Op", "Bank op code", "Instruction code"], f["bank_operation_code"]))
        self.r.shuffle(it)
        return [(self.lab(self.r.choice(labels)), v) for labels, v in it]

    def o_pipe_table(self, f, s):
        rows = ["| " + self.lab("Field") + " | " + self.lab("Value") + " |", "|---|---|"]
        rows += [f"| {k} | {v} |" for k, v in self._items(f, s)]
        if self.r.random() < 0.7:
            rows.append("")
            rows.append(self.remark())
        return "\n".join(rows)

    def o_kv_mixed(self, f, s):
        parts = [f"{k}={v}" for k, v in self._items(f, s)]
        out = "; ".join(parts)
        if self.r.random() < 0.6:
            out = self.remark() + " " + out if self.r.random() < 0.5 else out + " -- " + self.remark()
        return out

    def o_narrative(self, f, s):
        r = self.r
        oc, bn = f["ordering_customer"], f["beneficiary"]
        sents = [f"{self.ocr('This is')} {self.party_s(oc, ', ')} {self.ocr('writing')}."]
        sents.append(f"We would like to transfer {self.money(s)}"
                     + (f" to {self.party_s(bn, ', ')}" if s.get("missing") != "beneficiary" else "") + ".")
        if s.get("missing") != "value_date":
            sents.append(f"{self.ocr('Please make sure the funds arrive on')} {s['date']}.")
        sents.append(f"Quote {f['sender_reference']} as {self.ocr('our reference')}.")
        if f["ordering_institution"] or s.get("badbic", ("",))[0] == "ordering_institution":
            sents.append(f"Debit us through {self.bic_s(f, s, 'ordering_institution')}.")
        if f["account_with_institution"] or s.get("badbic", ("",))[0] == "account_with_institution":
            sents.append(f"Their bank's SWIFT code is {self.bic_s(f, s, 'account_with_institution')}.")
        if f["remittance_info"]:
            sents.append(f"It is for \"{f['remittance_info']}\".")
        if s.get("missing") != "charges":
            sents.append(f"{self.ocr('Charges')}: {s['charges']}.")
        if f["bank_operation_code"]:
            sents.append(f"Use op code {f['bank_operation_code']}.")
        head, rest = sents[:1], sents[1:]
        r.shuffle(rest)
        rest.insert(r.randint(0, len(rest)), self.remark())
        if r.random() < 0.5:
            rest.insert(r.randint(0, len(rest)), self.remark())
        return " ".join(head + rest)

    def o_swift_shuffled(self, f, s):
        r = self.r
        oc, bn = f["ordering_customer"], f["beneficiary"]
        t = (lambda tag: tag.lower()) if r.random() < 0.6 else (lambda tag: tag)
        ccy = s["ccy"].upper() if s["ccy"].isalpha() else (f["currency"] or s["ccy"])
        date = s["date"]
        if f["value_date"] and r.random() < 0.5:
            date = dt.date.fromisoformat(f["value_date"]).strftime("%y%m%d")
        blocks = [f":20:{f['sender_reference']}"]
        if f["bank_operation_code"]:
            blocks.append(f":{t('23B')}:{f['bank_operation_code']}")
        amt = s["amt"]
        if s.get("missing") != "value_date":
            blocks.append(f":{t('32A')}:{date} {ccy} {amt}")
        else:
            blocks.append(f":{t('32A')}:{ccy} {amt}")
        blocks.append(f":{t('50K')}:" + "\n".join(x for x in [("/" + oc["account"]) if oc["account"] else None, oc["name"], oc["address"]] if x))
        if f["ordering_institution"] or s.get("badbic", ("",))[0] == "ordering_institution":
            blocks.append(f":{t('52A')}:{self.bic_s(f, s, 'ordering_institution')}")
        if f["account_with_institution"] or s.get("badbic", ("",))[0] == "account_with_institution":
            blocks.append(f":{t('57A')}:{self.bic_s(f, s, 'account_with_institution')}")
        if s.get("missing") != "beneficiary":
            blocks.append(":59:" + "\n".join(x for x in [("/" + bn["account"]) if bn["account"] else None, bn["name"], bn["address"]] if x))
        if f["remittance_info"]:
            blocks.append(f":70:{f['remittance_info']}")
        if s.get("missing") != "charges":
            blocks.append(f":{t('71A')}:{s['charges'].upper()}")
        r.shuffle(blocks)
        if r.random() < 0.6:
            blocks.insert(r.randint(0, len(blocks)), "/* " + self.remark() + " */")
        return "\n".join(blocks)


def generate_ood(n: int, seed: int, exclude_texts: set[str] = frozenset()) -> list[dict]:
    g = OodGen(seed)
    rows, seen = [], set(exclude_texts)
    while len(rows) < n:
        ex = g.example(len(rows))
        h = hashlib.sha1(ex["text"].encode()).hexdigest()
        if h in seen:
            continue
        seen.add(h)
        rows.append(ex)
    return rows


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--n", type=int, default=100)
    ap.add_argument("--seed", type=int, default=20261123)
    ap.add_argument("--out", default=str(Path(__file__).parent / "ood_test.jsonl"))
    a = ap.parse_args()
    rows = generate_ood(a.n, a.seed)
    with open(a.out, "w", encoding="utf-8") as fh:
        for row in rows:
            fh.write(json.dumps(row, ensure_ascii=False) + "\n")
    styles = {}
    for row in rows:
        styles[row["ood_style"]] = styles.get(row["ood_style"], 0) + 1
    codes = {}
    for row in rows:
        for e in row["target"]["errors"]:
            codes[e.split(":")[0]] = codes.get(e.split(":")[0], 0) + 1
    manifest = {"synthetic": True, "note": "SYNTHETIC out-of-distribution test set (data/generate_ood.py); not real payments.",
                "seed": a.seed, "n": len(rows), "styles": styles, "error_codes": codes,
                "examples_with_errors": sum(1 for r in rows if r["target"]["errors"])}
    Path(a.out).with_suffix(".manifest.json").write_text(json.dumps(manifest, indent=2))
    print(json.dumps(manifest))


if __name__ == "__main__":
    assert all(c not in ISO4217 for c in OOD_BAD_CCY)
    assert all(not BIC_RE.match(b) for b in OOD_BAD_BICS)
    main()

In [ ]:
open('mt103/evals/__init__.py', 'w').close()

In [ ]:
%%writefile mt103/evals/run_eval.py
"""Shared evaluation for every model variant (base / fine-tuned / API reference / mock).

Examples:
  python evals/run_eval.py --backend ollama --model qwen2.5:0.5b-instruct --fewshot --name base-0.5b
  python evals/run_eval.py --backend ollama --model mt103-extractor --name ft-0.5b-smoke
  python evals/run_eval.py --backend claude-cli --model claude-haiku-5-5 --fewshot --name api-haiku
  python evals/run_eval.py --backend mock --limit 5 --name mock

Metrics: valid-JSON rate, schema-valid rate, field-level exact match (per field + macro),
error-flag precision/recall (micro over "CODE:field" keys, and example-level detection),
latency per example, output tokens/sec. Raw outputs are cached on disk
(results/cache/<name>.jsonl) so re-runs are free and resumable.
"""
from __future__ import annotations

import argparse
import concurrent.futures as cf
import datetime as dt
import hashlib
import json
import os
import re
import statistics
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
from prompts import FEWSHOT_IDS, chat_messages, ft_messages, parse_output, target_str  # noqa: E402
from postprocess import postprocess  # noqa: E402
from schema import Extraction, error_keys  # noqa: E402

SCORED_FIELDS = (
    "sender_reference", "bank_operation_code", "value_date", "currency", "amount",
    "ordering_customer.account", "ordering_customer.name", "ordering_customer.address",
    "ordering_institution", "account_with_institution",
    "beneficiary.account", "beneficiary.name", "beneficiary.address",
    "remittance_info", "charges",
)


# ---------------------------------------------------------------- scoring
def _get(fields, path):
    cur = fields
    for p in path.split("."):
        if not isinstance(cur, dict):
            return None
        cur = cur.get(p)
    return cur


def _norm(v):
    if v is None:
        return None
    if isinstance(v, str):
        s = re.sub(r"\s+", " ", v).strip().strip(".,;").casefold()
        return s or None
    return v


def field_match(path, gold, pred) -> bool:
    if path == "amount":
        if gold is None or pred is None:
            return gold is None and pred is None
        try:
            return abs(float(pred) - float(gold)) < 0.005
        except (TypeError, ValueError):
            return False
    if isinstance(pred, (int, float)) and not isinstance(pred, bool):
        pred = str(pred)
    return _norm(gold) == _norm(pred)


def score_one(gold: dict, pred_obj) -> dict:
    valid_json = isinstance(pred_obj, dict)
    schema_ok = False
    if valid_json:
        try:
            Extraction.model_validate(pred_obj)
            schema_ok = True
        except Exception:
            pass
    pf = pred_obj.get("fields") if valid_json and isinstance(pred_obj.get("fields"), dict) else {}
    per = {p: field_match(p, _get(gold["fields"], p), _get(pf, p)) for p in SCORED_FIELDS}
    gk = error_keys(gold["errors"])
    pk = error_keys(pred_obj.get("errors")) if valid_json else set()
    return {"valid_json": valid_json, "schema_valid": schema_ok, "fields": per,
            "err_tp": len(gk & pk), "err_fp": len(pk - gk), "err_fn": len(gk - pk),
            "has_err_gold": bool(gk), "has_err_pred": bool(pk)}


def aggregate(scores: list[dict], lat: list[float], toks: list[int], gen_s: list[float]) -> dict:
    n = len(scores)
    per = {p: sum(s["fields"][p] for s in scores) / n for p in SCORED_FIELDS}
    tp = sum(s["err_tp"] for s in scores)
    fp = sum(s["err_fp"] for s in scores)
    fn = sum(s["err_fn"] for s in scores)
    dtp = sum(s["has_err_gold"] and s["has_err_pred"] for s in scores)
    dfp = sum((not s["has_err_gold"]) and s["has_err_pred"] for s in scores)
    dfn = sum(s["has_err_gold"] and not s["has_err_pred"] for s in scores)
    div = lambda a, b: (a / b) if b else None  # noqa: E731
    return {
        "n": n,
        "valid_json_rate": sum(s["valid_json"] for s in scores) / n,
        "schema_valid_rate": sum(s["schema_valid"] for s in scores) / n,
        "field_accuracy_macro": sum(per.values()) / len(per),
        "field_accuracy": per,
        "all_fields_correct_rate": sum(all(s["fields"].values()) for s in scores) / n,
        "error_precision": div(tp, tp + fp), "error_recall": div(tp, tp + fn),
        "error_detect_precision": div(dtp, dtp + dfp), "error_detect_recall": div(dtp, dtp + dfn),
        "latency_mean_s": statistics.mean(lat) if lat else None,
        "latency_p50_s": statistics.median(lat) if lat else None,
        "latency_p95_s": (sorted(lat)[max(0, int(round(0.95 * len(lat))) - 1)] if lat else None),
        "tokens_per_sec": (sum(toks) / sum(gen_s)) if gen_s and sum(gen_s) > 0 else None,
    }


# ---------------------------------------------------------------- backends
def call_ollama(msgs, model, url, num_ctx):
    body = json.dumps({"model": model, "messages": msgs, "stream": False,
                       "options": {"temperature": 0, "num_predict": 600, "num_ctx": num_ctx, "num_thread": 1,
                                   "seed": 0}}).encode()
    req = urllib.request.Request(url.rstrip("/") + "/api/chat", data=body,
                                 headers={"Content-Type": "application/json"})
    t0 = time.time()
    with urllib.request.urlopen(req, timeout=900) as r:
        out = json.loads(r.read())
    lat = time.time() - t0
    return {"raw": out["message"]["content"], "latency_s": lat,
            "out_tokens": out.get("eval_count", 0), "gen_s": out.get("eval_duration", 0) / 1e9,
            "prompt_tokens": out.get("prompt_eval_count", 0)}


def call_claude_cli(msgs, model):
    """Uses the logged-in `claude` CLI (no API key on this host). Few-shot is inlined."""
    sys_prompt = msgs[0]["content"]
    convo = []
    for m in msgs[1:-1]:
        tag = "INPUT" if m["role"] == "user" else "OUTPUT"
        convo.append(f"<{tag}>\n{m['content']}\n</{tag}>")
    prompt = ("Examples:\n" + "\n".join(convo) + "\n\n" if convo else "") + \
             f"<INPUT>\n{msgs[-1]['content']}\n</INPUT>\nReply with the OUTPUT JSON only."
    env = dict(os.environ, ENABLE_CLAUDEAI_MCP_SERVERS="false", CLAUDE_CODE_DISABLE_NONESSENTIAL_TRAFFIC="1")
    cmd = ["claude", "-p", "--model", model, "--output-format", "json", "--tools", "",
           "--setting-sources", "", "--strict-mcp-config", "--no-session-persistence",
           "--system-prompt", sys_prompt, prompt]
    t0 = time.time()
    p = subprocess.run(cmd, capture_output=True, text=True, timeout=300, env=env)
    lat = time.time() - t0
    if p.returncode != 0:
        raise RuntimeError(f"claude CLI failed: {p.stderr[-300:]}")
    out = json.loads(p.stdout)
    usage = out.get("usage") or {}
    api_ms = out.get("duration_api_ms") or 0
    return {"raw": out.get("result", ""), "latency_s": lat, "out_tokens": usage.get("output_tokens", 0),
            "gen_s": api_ms / 1000.0, "prompt_tokens": usage.get("input_tokens", 0)}


def call_mock(msgs, ex):
    """Deterministic mock for CI: returns the gold target, except it corrupts every 5th example."""
    raw = target_str(ex["target"])
    h = int(hashlib.md5(ex["id"].encode()).hexdigest(), 16)
    if h % 5 == 0:
        raw = "Sure! " + raw[: len(raw) // 2]  # truncated -> invalid JSON
    return {"raw": raw, "latency_s": 0.001, "out_tokens": len(raw) // 4, "gen_s": 0.001, "prompt_tokens": 0}


# ---------------------------------------------------------------- main
def load_jsonl(p):
    with open(p, encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]


def run(args) -> dict:
    test = load_jsonl(args.data)[: args.limit] if args.limit else load_jsonl(args.data)
    fewshot = []
    if args.fewshot:
        train = {r["id"]: r for r in load_jsonl(ROOT / "data" / "train.jsonl")}  # fixed v1 few-shot ids
        fewshot = [train[i] for i in FEWSHOT_IDS]
    cache_dir = Path(args.out_dir) / "cache"
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_p = cache_dir / f"{args.name}.jsonl"
    cache = {}
    if cache_p.exists() and not args.no_cache:
        for r in load_jsonl(cache_p):
            cache[r["key"]] = r

    def key(ex, msgs):
        model_id = args.model + (f"+{args.adapter}" if args.adapter else "")
        return hashlib.sha1(json.dumps([args.backend, model_id, msgs], ensure_ascii=False).encode()).hexdigest()

    def msgs_for(ex):
        return ft_messages(ex["text"]) if args.prompt == "ft" else chat_messages(ex["text"], fewshot)

    if args.backend == "hf" and not args.cache_only:  # batched GPU generation up front, into the cache
        todo = [(ex, msgs_for(ex)) for ex in test]
        todo = [(ex, m, key(ex, m)) for ex, m in todo if key(ex, m) not in cache]
        if todo:
            from evals.hf_backend import generate, load
            model, tok = load(args.model, args.adapter)
            outs = generate(model, tok, [m for _, m, _ in todo], batch_size=args.batch_size)
            with open(cache_p, "a", encoding="utf-8") as fh:
                for (ex, _, k), o in zip(todo, outs):
                    cache[k] = {"key": k, "id": ex["id"], **o}
                    fh.write(json.dumps(cache[k], ensure_ascii=False) + "\n")

    def work(ex):
        msgs = msgs_for(ex)
        k = key(ex, msgs)
        if k in cache:
            return ex, cache[k], True
        if args.cache_only:  # scored as a failed call (no output), exactly like the original run's failures
            return ex, {"key": k, "id": ex["id"], "raw": None, "latency_s": None, "out_tokens": 0, "gen_s": 0,
                        "error": "not in cache (--cache-only)"}, True
        for attempt in range(3):
            try:
                if args.backend == "ollama":
                    res = call_ollama(msgs, args.model, args.ollama_url, args.num_ctx)
                elif args.backend == "claude-cli":
                    res = call_claude_cli(msgs, args.model)
                elif args.backend == "hf":
                    raise RuntimeError("hf output missing from the batched pre-pass")
                else:
                    res = call_mock(msgs, ex)
                break
            except Exception as e:  # noqa: BLE001
                if attempt == 2:
                    res = {"raw": None, "latency_s": None, "out_tokens": 0, "gen_s": 0, "error": str(e)[:300]}
                time.sleep(2 * (attempt + 1))
        res = {"key": k, "id": ex["id"], **res}
        return ex, res, False

    rows = []
    workers = args.workers if args.backend == "claude-cli" else 1
    with open(cache_p, "a", encoding="utf-8") as cfh, cf.ThreadPoolExecutor(workers) as pool:
        for i, (ex, res, hit) in enumerate(pool.map(work, test), 1):
            if not hit and args.backend != "mock" and res.get("raw") is not None:
                cfh.write(json.dumps(res, ensure_ascii=False) + "\n")
                cfh.flush()
            rows.append((ex, res))
            if i % 10 == 0 or i == len(test):
                print(f"[{args.name}] {i}/{len(test)}", flush=True)

    scores, lat, toks, gen = [], [], [], []
    per_example = []
    for ex, res in rows:
        obj = parse_output(res.get("raw"))
        if args.postprocess:
            obj = postprocess(obj, ex["text"])
        s = score_one(ex["target"], obj)
        scores.append(s)
        if res.get("latency_s") is not None:
            lat.append(res["latency_s"])
        if res.get("gen_s"):
            toks.append(res.get("out_tokens", 0))
            gen.append(res["gen_s"])
        per_example.append({"id": ex["id"], "valid_json": s["valid_json"], "schema_valid": s["schema_valid"],
                            "fields_wrong": [k for k, v in s["fields"].items() if not v],
                            "err_fp": s["err_fp"], "err_fn": s["err_fn"]})
    summary = aggregate(scores, lat, toks, gen)
    name = args.name + ("+pp" if args.postprocess else "")
    summary.update({"name": name, "postprocess": bool(args.postprocess), "backend": args.backend, "model": args.model,
                    "fewshot": bool(args.fewshot), "prompt": args.prompt, "data": str(args.data),
                    "date": dt.date.today().isoformat(), "command": " ".join(sys.argv)})
    out = Path(args.out_dir) / f"eval_{name}.json"
    out.write_text(json.dumps({"summary": summary, "per_example": per_example}, indent=2, ensure_ascii=False))
    print(json.dumps({k: v for k, v in summary.items() if k != "field_accuracy"}, indent=2))
    return summary


def parse_args(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--backend", choices=["ollama", "claude-cli", "mock", "hf"], required=True,
                    help="hf = transformers on GPU/CPU (--model = HF id or path, optional --adapter LoRA dir)")
    ap.add_argument("--model", default="")
    ap.add_argument("--name", required=True)
    ap.add_argument("--data", default=str(ROOT / "data" / "test.jsonl"))
    ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--fewshot", action="store_true")
    ap.add_argument("--prompt", choices=["full", "ft"], default="full",
                    help="full = long instruction prompt (+ few-shot); ft = short prompt the LoRA model was trained with")
    ap.add_argument("--ollama-url", default=os.environ.get("OLLAMA_URL", "http://127.0.0.1:21434"))
    ap.add_argument("--num-ctx", type=int, default=4096)
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--out-dir", default=str(ROOT / "results"))
    ap.add_argument("--no-cache", action="store_true")
    ap.add_argument("--postprocess", action="store_true",
                    help="apply postprocess.py (deterministic rules) before scoring; reads the same cache, "
                         "writes eval_<name>+pp.json")
    ap.add_argument("--adapter", default="", help="hf backend: LoRA adapter directory")
    ap.add_argument("--batch-size", type=int, default=16, help="hf backend: generation batch size")
    ap.add_argument("--cache-only", action="store_true", help="never call a model; a cache miss is scored as a failed call")
    return ap.parse_args(argv)


if __name__ == "__main__":
    run(parse_args())

In [ ]:
%%writefile mt103/evals/hf_backend.py
"""Batched greedy generation with Hugging Face transformers (+ optional LoRA adapter), for GPU evals.

Used by `evals/run_eval.py --backend hf` (for example in the Kaggle kernel right after training), so the
fine-tuned adapter is scored by exactly the same scorer as the Ollama runs. Outputs are written to the
same results/cache/<name>.jsonl format.
"""
from __future__ import annotations

import time


def load(base: str, adapter: str | None = None):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(adapter or base)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    gpu = torch.cuda.is_available()
    model = AutoModelForCausalLM.from_pretrained(base, dtype=torch.float16 if gpu else torch.float32,
                                                 device_map="auto" if gpu else None)
    if adapter:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, adapter)
    model.eval()
    return model, tok


def generate(model, tok, conversations: list[list[dict]], batch_size: int = 16, max_new_tokens: int = 600):
    """Greedy-decode each conversation; returns [{raw, latency_s, out_tokens, gen_s}] in input order.
    latency_s is the batch wall time divided by the batch size (per-example latency is not comparable to
    the single-request CPU numbers)."""
    import torch
    out = []
    order = sorted(range(len(conversations)), key=lambda i: len(str(conversations[i])))  # less padding
    res = [None] * len(conversations)
    for s in range(0, len(order), batch_size):
        idx = order[s:s + batch_size]
        prompts = [tok.apply_chat_template(conversations[i], add_generation_prompt=True, tokenize=False) for i in idx]
        enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
        t0 = time.time()
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                 pad_token_id=tok.pad_token_id)
        dt = time.time() - t0
        new = gen[:, enc["input_ids"].shape[1]:]
        for j, i in enumerate(idx):
            ids = new[j]
            n = int((ids != tok.pad_token_id).sum())
            res[i] = {"raw": tok.decode(ids, skip_special_tokens=True), "latency_s": dt / len(idx),
                      "out_tokens": n, "gen_s": dt / len(idx), "prompt_tokens": int(enc["attention_mask"][j].sum())}
    out.extend(res)
    return out

In [ ]:
%%writefile mt103/train/config.yaml
# LoRA config. `mode` is chosen automatically (full if CUDA is available, else smoke)
# unless overridden with --mode.
base_model_cpu: Qwen/Qwen2.5-0.5B-Instruct
base_model_gpu: Qwen/Qwen2.5-1.5B-Instruct
lora:
  r: 16
  alpha: 32
  dropout: 0.05
  target_modules: [q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj]
train:
  lr: 2.0e-4
  scheduler: cosine
  warmup_ratio: 0.05
  max_length: 1024
  seed: 103
full:            # GPU: QLoRA 4-bit, 3 epochs over 2400 train examples
  epochs: 3
  batch_size: 8
  grad_accum: 2
  val_loss_examples: 300
  val_gen_examples: 50
  qlora: true
smoke:           # CPU: prove the pipeline end to end (NOT a fully trained model)
  max_steps: 150
  batch_size: 1
  grad_accum: 2          # effective batch 2 -> 300 training examples seen
  gradient_checkpointing: true   # keeps RAM under ~2.5 GB on the shared VPS
  eval_every: 50
  val_loss_examples: 40
  val_gen_examples: 20
  qlora: false

In [ ]:
%%writefile mt103/train/train_lora.py
"""LoRA / QLoRA fine-tuning of Qwen2.5-Instruct for MT103 extraction (TRL SFTTrainer).

    python train/train_lora.py                 # auto: full (GPU, QLoRA 4-bit) or smoke (CPU)
    python train/train_lora.py --mode smoke    # ~150 steps on CPU: pipeline proof, NOT fully trained

Outputs:
    models/adapter/                 LoRA adapter (+ tokenizer)
    results/train_log.json          loss per step, val loss per eval, val field accuracy
    results/loss.png                training / validation loss curve
"""
from __future__ import annotations

import argparse
import json
import os
import sys
import time
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))

import yaml  # noqa: E402

from prompts import ft_messages, parse_output, target_str  # noqa: E402


def load_jsonl(p):
    with open(p, encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]


def to_prompt_completion(rows):
    """Conversational prompt/completion format: TRL computes loss on the completion only."""
    return [{"prompt": ft_messages(r["text"]),
             "completion": [{"role": "assistant", "content": target_str(r["target"])}]} for r in rows]


def val_generation_accuracy(model, tok, rows, max_new_tokens=400):
    """Greedy-decode val examples and score them with the same scorer as evals/run_eval.py."""
    import torch
    from evals.run_eval import aggregate, score_one
    model.eval()
    scores, lat, toks, gen = [], [], [], []
    for r in rows:
        ids = tok.apply_chat_template(ft_messages(r["text"]), add_generation_prompt=True, return_tensors="pt",
                                      return_dict=True)
        ids = {k: v.to(model.device) for k, v in ids.items()}
        t0 = time.time()
        with torch.no_grad():
            out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False,
                                 pad_token_id=tok.eos_token_id)
        dt_ = time.time() - t0
        new = out[0, ids["input_ids"].shape[1]:]
        raw = tok.decode(new, skip_special_tokens=True)
        scores.append(score_one(r["target"], parse_output(raw)))
        lat.append(dt_); toks.append(len(new)); gen.append(dt_)
    model.train()
    s = aggregate(scores, lat, toks, gen)
    return {k: s[k] for k in ("n", "valid_json_rate", "schema_valid_rate", "field_accuracy_macro",
                              "error_precision", "error_recall")}


def plot_loss(log, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 4), dpi=120)
    tr = [(e["step"], e["loss"]) for e in log["history"] if "loss" in e]
    va = [(e["step"], e["eval_loss"]) for e in log["history"] if "eval_loss" in e]
    if tr:
        ax.plot(*zip(*tr), color="#2a6fdb", lw=1.6, label="train loss")
    if va:
        ax.plot(*zip(*va), color="#d9534f", lw=1.6, marker="o", label="val loss")
    ax.set_xlabel("step"); ax.set_ylabel("loss (completion tokens)")
    ax.set_title(f"LoRA on {log['base_model'].split('/')[-1]} — {log['mode']} run" +
                 (" (CPU smoke, not fully trained)" if log["mode"] == "smoke" else ""), fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=0.25); ax.legend(frameon=False)
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default=str(ROOT / "train" / "config.yaml"))
    ap.add_argument("--mode", choices=["auto", "full", "smoke"], default="auto")
    ap.add_argument("--base-model", default=None, help="HF id or local path (overrides config)")
    ap.add_argument("--data-dir", default=str(ROOT / "data"))
    ap.add_argument("--out", default=str(ROOT / "models" / "adapter"))
    ap.add_argument("--results", default=str(ROOT / "results"))
    ap.add_argument("--max-steps", type=int, default=None)
    ap.add_argument("--val-gen-examples", type=int, default=None)
    ap.add_argument("--epochs", type=float, default=None, help="full mode: overrides the config's epochs")
    ap.add_argument("--no-qlora", action="store_true", help="full mode: plain fp16 LoRA instead of 4-bit QLoRA")
    ap.add_argument("--batch-size", type=int, default=None, help="overrides the config's per-device batch size")
    ap.add_argument("--grad-accum", type=int, default=None, help="overrides the config's gradient accumulation")
    a = ap.parse_args()

    import torch
    from datasets import Dataset
    from peft import LoraConfig
    from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback
    from trl import SFTConfig, SFTTrainer

    cfg = yaml.safe_load(open(a.config))
    gpu = torch.cuda.is_available()
    mode = a.mode if a.mode != "auto" else ("full" if gpu else "smoke")
    mc = dict(cfg[mode])
    if a.no_qlora:
        mc["qlora"] = False
    if a.batch_size:
        mc["batch_size"] = a.batch_size
    if a.grad_accum:
        mc["grad_accum"] = a.grad_accum
    if not gpu:
        torch.set_num_threads(int(os.environ.get("OMP_NUM_THREADS", "1")))
    base = a.base_model or (cfg["base_model_gpu"] if gpu else cfg["base_model_cpu"])
    local = ROOT / "models" / "base-qwen2.5-0.5b-instruct"
    if a.base_model is None and not gpu and local.exists():
        base = str(local)
    print(f"mode={mode} gpu={gpu} base={base}", flush=True)

    tok = AutoTokenizer.from_pretrained(base)
    kw = {}
    if gpu and mc.get("qlora"):
        from transformers import BitsAndBytesConfig
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_compute_dtype=torch.float16,
                                                       bnb_4bit_use_double_quant=True)
        kw["device_map"] = "auto"
    # CPU: bf16 weights (the VPS has avx512_bf16) to halve memory; LoRA weights stay fp32 (peft default).
    model = AutoModelForCausalLM.from_pretrained(base, dtype=torch.float16 if gpu else torch.bfloat16, **kw)
    if gpu and mc.get("qlora"):
        from peft import prepare_model_for_kbit_training
        model = prepare_model_for_kbit_training(model)

    train_rows = load_jsonl(Path(a.data_dir) / "train.jsonl")
    val_rows = load_jsonl(Path(a.data_dir) / "val.jsonl")
    train_ds = Dataset.from_list(to_prompt_completion(train_rows)).shuffle(seed=cfg["train"]["seed"])
    val_ds = Dataset.from_list(to_prompt_completion(val_rows[: mc["val_loss_examples"]]))
    gen_rows = val_rows[: (a.val_gen_examples if a.val_gen_examples is not None else mc["val_gen_examples"])]

    lc = cfg["lora"]
    peft_cfg = LoraConfig(r=lc["r"], lora_alpha=lc["alpha"], lora_dropout=lc["dropout"],
                          target_modules=lc["target_modules"], task_type="CAUSAL_LM")
    tc = cfg["train"]
    steps_kw = ({"num_train_epochs": a.epochs or mc["epochs"]} if mode == "full"
                else {"max_steps": a.max_steps or mc["max_steps"]})
    eval_kw = ({"eval_strategy": "epoch"} if mode == "full" else
               {"eval_strategy": "steps", "eval_steps": mc["eval_every"]})
    sft = SFTConfig(
        output_dir=str(ROOT / "train" / "runs" / mode), per_device_train_batch_size=mc["batch_size"],
        per_device_eval_batch_size=mc["batch_size"], gradient_accumulation_steps=mc["grad_accum"],
        learning_rate=tc["lr"], lr_scheduler_type=tc["scheduler"], warmup_steps=tc["warmup_ratio"],
        max_length=tc["max_length"], logging_steps=5, save_strategy="no", report_to=[],
        seed=tc["seed"], fp16=gpu, bf16=not gpu, use_cpu=not gpu, dataloader_num_workers=0,
        gradient_checkpointing=bool(mc.get("gradient_checkpointing", False)),
        **steps_kw, **eval_kw)

    log = {"mode": mode, "base_model": base, "gpu": gpu, "lora": lc, "train": tc, "mode_cfg": mc,
           "n_train": len(train_rows), "history": [], "val_generation": []}

    class GenEval(TrainerCallback):
        """Full mode: generation-based val accuracy after every epoch."""
        def on_epoch_end(self, args, state, control, model=None, **k):
            if mode == "full":
                m = val_generation_accuracy(model, tok, gen_rows)
                m.update({"epoch": state.epoch, "step": state.global_step})
                log["val_generation"].append(m)
                print("val_generation", m, flush=True)

    trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=val_ds,
                         processing_class=tok, peft_config=peft_cfg, callbacks=[GenEval()])
    t0 = time.time()
    trainer.train()
    log["train_seconds"] = round(time.time() - t0, 1)
    log["history"] = trainer.state.log_history
    if mode == "smoke":  # one generation-based val score at the end
        m = val_generation_accuracy(trainer.model, tok, gen_rows)
        m.update({"epoch": trainer.state.epoch, "step": trainer.state.global_step})
        log["val_generation"].append(m)
        print("val_generation", m, flush=True)

    out = Path(a.out)
    trainer.model.save_pretrained(out)
    tok.save_pretrained(out)
    (out / "TRAINING_INFO.json").write_text(json.dumps(
        {"mode": mode, "base_model": base if mode == "full" else cfg["base_model_cpu"],
         "steps": trainer.state.global_step, "epochs": trainer.state.epoch,
         "label": ("CPU smoke run, not fully trained" if mode == "smoke"
                   else "full QLoRA run" if mc.get("qlora") else "full LoRA run (fp16)"),
         "data_dir": str(a.data_dir), "n_train": len(train_rows)}, indent=2))
    res = Path(a.results); res.mkdir(exist_ok=True)
    (res / "train_log.json").write_text(json.dumps(log, indent=2, default=str))
    plot_loss(log, res / "loss.png")
    print(f"saved adapter to {out} in {log['train_seconds']}s", flush=True)


if __name__ == "__main__":
    main()

## Data: SYNTHETIC, regenerated from the seeds, checked against the repo copies

In [ ]:
%cd /kaggle/working/mt103
!python data/generate.py --n 3000 --seed 103 --out data
!python data/generate_v2.py --n 6000 --seed 2026 --out data/v2
!python data/generate_ood.py --n 100 --seed 20261123 --out data/ood_test.jsonl
import hashlib
EXPECTED = {
 "data/test.jsonl": "3b488ecd75ae7c8d72ebd88a3db6bcd760965621486d9cc57ebb4a1e6715f845",
 "data/ood_test.jsonl": "5d1e2a2a6d9ef6a23be2e19ac72cde03c54d35f5a57e1a5d86a3d9cb53b80ae7",
 "data/v2/train.jsonl": "d069f6bfc9355ea022c9a5d6b6ba62ddacfa91b955fa7b9e403864ec1a4114dd",
 "data/v2/val.jsonl": "3c2ffcc1b941825a15403e226c67ecafeed860903349683742ee18e75cd7ed7a",
 "data/v2/test.jsonl": "fabfd01510e44159a6b90be1cb0d43b9836aca133eb0dafe8b826aa5e963261e"
}
bad = [p for p, h in EXPECTED.items() if hashlib.sha256(open(p, "rb").read()).hexdigest() != h]
assert not bad, f"regenerated data differs from the repo copy: {bad}"
print("all data files match the repo (SHA-256)")

## Train (LoRA on data/v2; val loss + val field accuracy logged every epoch)

In [ ]:
!python train/train_lora.py --mode full --base-model {BASE_MODEL} --data-dir data/v2 --epochs {EPOCHS} {EXTRA_TRAIN_ARGS} --out models/adapter --results results
import json
log = json.load(open("results/train_log.json"))
print("train seconds:", log["train_seconds"])
for v in log["val_generation"]:
    print(v)
from IPython.display import Image, display
display(Image("results/loss.png"))

## Evaluate: v1 test, v2 test, OOD; raw and + postprocess.py

In [ ]:
RUN = "kaggle-ft-qwen2.5-0.5b-lora-v2"
SETS = {"": "data/test.jsonl", "v2-": "data/v2/test.jsonl", "ood-": "data/ood_test.jsonl"}
for prefix, path in SETS.items():
    base = f"python evals/run_eval.py --backend hf --model {BASE_MODEL} --adapter models/adapter --prompt ft --data {path} --name {prefix}{RUN}"
    !{base} --batch-size {EVAL_BATCH} 2>&1 | tail -n 3
    !{base} --postprocess --cache-only 2>&1 | tail -n 3
import json, glob
metrics = {}
for p in sorted(glob.glob(f"results/eval_*{RUN}*.json")):
    s = json.load(open(p))["summary"]
    metrics[s["name"]] = {k: s[k] for k in ("n", "valid_json_rate", "schema_valid_rate", "field_accuracy_macro",
                                             "all_fields_correct_rate", "error_precision", "error_recall",
                                             "field_accuracy", "data")}
    print(f"{s['name']:55s} n={s['n']:4d} field acc={100 * s['field_accuracy_macro']:.1f}%  "
          f"err P/R={s['error_precision']}/{s['error_recall']}")
json.dump({"base_model": BASE_MODEL, "epochs": EPOCHS, "train_args": EXTRA_TRAIN_ARGS,
           "gpu": torch.cuda.get_device_name(0), "train_seconds": log["train_seconds"],
           "val_generation": log["val_generation"], "evals": metrics},
          open("/kaggle/working/metrics.json", "w"), indent=2)

## Package outputs

In [ ]:
!cd models && zip -qr /kaggle/working/adapter.zip adapter && cd ..
!zip -qr /kaggle/working/results.zip results
!ls -lh /kaggle/working/adapter.zip /kaggle/working/results.zip /kaggle/working/metrics.json
print("Download adapter.zip, results.zip and metrics.json from the Output tab (or `kaggle kernels output`).")